In [ ]:
import os, sys, json, math, time, random, hashlib, warnings, itertools, shutil, glob, re
from pathlib import Path
from collections import defaultdict, OrderedDict, deque

warnings.filterwarnings("ignore")


class CFG:
    PROTOCOL = "v2"          # mixed into every experiment hash; bump to invalidate caches

    # ---------------- paths ----------------
    DATA_SEARCH_ROOT = "/kaggle/input/datasets/syedmuntasinfayaz/fnirs2mw-dataset/fNIRS2MW_dataset"
    WORK_DIR         = "/kaggle/working"
    WINDOW_TAG       = "size_30sec_150ts_stride_03ts"

    # ---------------- task ----------------
    TASK_LABELS = [0, 2]
    N_CLASSES   = 2
    N_TIMESTEPS = 150

    # ---------------- protocol ----------------
    SEED  = 2026
    # 12 repeated leave-K-subjects-out replicates. Each replicate redraws the
    # held-out set, so N for the paired tests comes from independent subjects
    # rather than from three lucky splits.
    SEEDS = list(range(2026, 2038))

    N_TEST_SUBJECTS = 14
    N_VAL_SUBJECTS  = 16      # was 8: the val curve was pure noise at that size
    TRAIN_FRAC      = 0.70
    VAL_FRAC        = 0.10
    GUARD_CHUNKS    = 50

    # ---------------- optimisation ----------------
    ROUNDS         = 60
    LOCAL_EPOCHS   = 2
    CLIENT_FRAC    = 0.40
    BATCH_SIZE     = 64
    LR             = 3e-4     # was 1e-3: val peaked at round 2 then collapsed
    WEIGHT_DECAY   = 1e-4
    FEDPROX_MU     = 0.01
    GRAD_CLIP      = 1.0
    EARLY_PATIENCE = 25       # was 15
    VAL_SMOOTH     = 3        # rounds of moving average before model selection

    # ---------------- personalization ----------------
    CALIB_BUDGETS  = [0, 8, 16, 32, 64, 128]
    CALIB_EPOCHS   = 20
    CALIB_LR       = 5e-4

    # ---------------- models ----------------
    DEFAULT_MODEL  = "fnirsnet"
    MODEL_ZOO      = ["fnirsnet", "deepconv", "gru", "tinytx"]
    F1, D_MULT, F2 = 32, 2, 64   # was 16/2/32: FedBN had 320 personal params to work with

    # ---------------- engineering ----------------
    CKPT_EVERY     = 5
    NUM_WORKERS    = 2
    AMP            = True
    VERBOSE        = 1

    # ---------------- ablation switches ----------------
    RUN_ABLATIONS  = True
    RUN_MODEL_ZOO  = True
    RUN_DP         = True
    # widened: at [0, 0.001 ... 0.05] the ordering was random, i.e. the noise
    # never reached the parameter scale and the ablation showed nothing
    DP_SIGMAS      = [0.0, 0.05, 0.1, 0.25, 0.5, 1.0]

    ABL_REPS       = 3        # replicates per ablation setting


CFG.CACHE_DIR  = f"{CFG.WORK_DIR}/cache"
CFG.CKPT_DIR   = f"{CFG.WORK_DIR}/checkpoints"
CFG.RESULT_DIR = f"{CFG.WORK_DIR}/results"
CFG.FIG_DIR    = f"{CFG.WORK_DIR}/figures"
CFG.TAB_DIR    = f"{CFG.WORK_DIR}/tables"

for _d in [CFG.CACHE_DIR, CFG.CKPT_DIR, CFG.RESULT_DIR, CFG.FIG_DIR, CFG.TAB_DIR]:
    Path(_d).mkdir(parents=True, exist_ok=True)

print(f"protocol {CFG.PROTOCOL} | {len(CFG.SEEDS)} split replicates | "
      f"val subjects {CFG.N_VAL_SUBJECTS} | lr {CFG.LR}")


In [ ]:
# ============================================================
# 01 | ENVIRONMENT, SEEDING, DEVICE
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import scipy.stats as st
from sklearn.metrics import (balanced_accuracy_score, accuracy_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve, cohen_kappa_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(CFG.SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = CFG.AMP and DEVICE.type == "cuda"

print("torch      :", torch.__version__)
print("device     :", DEVICE)
if DEVICE.type == "cuda":
    print("gpu        :", torch.cuda.get_device_name(0))
    print("vram (GB)  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print("amp        :", USE_AMP)


In [ ]:
# ============================================================
# 02 | FIGURE STYLE (flat vector, muted academic palette)
# ============================================================
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# role-coded palette: spatial=blue, spectral=purple, attention=teal,
# classification=coral, neutral=gray, accent=ochre
PAL = {
    "blue":    "#3B6EA5",
    "purple":  "#7B5AA6",
    "teal":    "#3F8F8A",
    "coral":   "#D1755B",
    "gray":    "#7A7A7A",
    "ochre":   "#B08A3E",
    "dark":    "#2B2B2B",
    "light":   "#D9D9D9",
}
PAL_SEQ = [PAL["blue"], PAL["purple"], PAL["teal"], PAL["coral"], PAL["gray"], PAL["ochre"]]

METHOD_COLOR = {
    "Local-only":  PAL["gray"],
    "FedAvg":      PAL["blue"],
    "FedProx":     PAL["purple"],
    "FedBN":       PAL["teal"],
    "FedPer":      PAL["coral"],
    "Centralized": PAL["ochre"],
    "LR-feat":     "#9AA5B1",
    "RF-feat":     "#5E6B78",
}

plt.rcParams.update({
    "figure.dpi":          130,
    "savefig.dpi":         300,
    "savefig.bbox":        "tight",
    "savefig.pad_inches":  0.02,
    "font.family":         "serif",
    "font.serif":          ["DejaVu Serif", "Times New Roman", "Nimbus Roman"],
    "font.size":           9,
    "axes.titlesize":      9.5,
    "axes.labelsize":      9,
    "xtick.labelsize":     8,
    "ytick.labelsize":     8,
    "legend.fontsize":     8,
    "axes.linewidth":      0.8,
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "axes.grid":           True,
    "grid.color":          "#E4E4E4",
    "grid.linewidth":      0.6,
    "grid.linestyle":      "-",
    "axes.axisbelow":      True,
    "lines.linewidth":     1.3,
    "legend.frameon":      False,
    "pdf.fonttype":        42,
    "ps.fonttype":         42,
})

# single-column 3.35 in, double-column 7.0 in (IEEE / Elsevier convention)
COL1, COL2 = 3.35, 7.0


def savefig(fig, name, tight=True):
    if tight:
        fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(f"{CFG.FIG_DIR}/{name}.{ext}")
    plt.show(fig)
    print(f"  saved figure -> {name}.pdf / .png")


In [ ]:
# ============================================================
# 03 | RESUMABLE EXPERIMENT REGISTRY + CHECKPOINT UTILITIES
# ============================================================

def cfg_hash(d):
    # deterministic short id for an experiment configuration
    payload = json.dumps(d, sort_keys=True, default=str)
    return hashlib.md5(payload.encode()).hexdigest()[:10]


def result_path(exp):
    return Path(CFG.RESULT_DIR) / f"{exp}.json"


def has_result(exp):
    return result_path(exp).exists()


def load_result(exp):
    with open(result_path(exp)) as f:
        return json.load(f)


def save_result(exp, record):
    # atomic write so an interrupted kernel cannot leave a truncated record
    p = result_path(exp)
    tmp = p.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(record, f, indent=2, default=str)
    tmp.replace(p)


def all_results():
    rows = []
    for p in sorted(Path(CFG.RESULT_DIR).glob("*.json")):
        try:
            with open(p) as f:
                rows.append(json.load(f))
        except Exception:
            continue
    return rows


def ckpt_path(exp):
    return Path(CFG.CKPT_DIR) / f"{exp}.pt"


def save_ckpt(exp, state):
    p = ckpt_path(exp)
    tmp = p.with_suffix(".tmp")
    torch.save(state, tmp)
    tmp.replace(p)


def load_ckpt(exp):
    p = ckpt_path(exp)
    if not p.exists():
        return None
    try:
        return torch.load(p, map_location="cpu")
    except Exception as e:
        print(f"  [warn] corrupt checkpoint {p.name}: {e}. Starting fresh.")
        return None


def clear_ckpt(exp):
    p = ckpt_path(exp)
    if p.exists():
        p.unlink()


def to_latex(df, name, caption, label, float_fmt="%.3f", escape=False):
    tex = df.to_latex(index=False, float_format=float_fmt, escape=escape,
                      caption=caption, label=label, column_format="l" + "c" * (df.shape[1] - 1))
    with open(f"{CFG.TAB_DIR}/{name}.tex", "w") as f:
        f.write(tex)
    print(f"  saved table -> {name}.tex")
    return tex


print("registry ready. completed experiments on disk:", len(list(Path(CFG.RESULT_DIR).glob('*.json'))))


In [ ]:
# ============================================================
# 04 | LOCATE THE SLIDING-WINDOW DIRECTORY AND SNIFF THE SCHEMA
# ============================================================

def find_window_dir(root, tag):
    hits = []
    for dirpath, dirnames, filenames in os.walk(root):
        if os.path.basename(dirpath) == tag:
            hits.append(dirpath)
    if not hits:
        # fall back: any directory that looks like a sliding-window release
        for dirpath, dirnames, filenames in os.walk(root):
            if re.search(r"size_\d+sec_\d+ts_stride_\d+ts", os.path.basename(dirpath)):
                hits.append(dirpath)
    return sorted(hits, key=len)


CANDIDATES = find_window_dir(CFG.DATA_SEARCH_ROOT, CFG.WINDOW_TAG)
if not CANDIDATES:
    raise FileNotFoundError(
        f"Could not find '{CFG.WINDOW_TAG}' under {CFG.DATA_SEARCH_ROOT}. "
        "Set CFG.DATA_SEARCH_ROOT to the mount point of your Kaggle dataset."
    )

WIN_DIR = CANDIDATES[0]
CSV_FILES = sorted(glob.glob(os.path.join(WIN_DIR, "**", "*.csv"), recursive=True))
print("window dir :", WIN_DIR)
print("csv files  :", len(CSV_FILES))
print("example    :", os.path.basename(CSV_FILES[0]) if CSV_FILES else "NONE")

if not CSV_FILES:
    raise FileNotFoundError("No CSV files found under the window directory.")

_probe = pd.read_csv(CSV_FILES[0], nrows=2000)
print("\ncolumns:", list(_probe.columns))
print(_probe.head(3))


In [ ]:
# ============================================================
# 05 | SCHEMA RESOLUTION
# ============================================================

def resolve_schema(df):
    cols = list(df.columns)
    lower = {c: c.lower().strip() for c in cols}

    label_col = next((c for c in cols if lower[c] in
                      ("label", "labels", "y", "target", "class", "nback", "n_back")), None)
    chunk_col = next((c for c in cols if lower[c] in
                      ("chunk", "chunk_id", "window", "window_id", "segment", "trial", "chunk_num")), None)

    if label_col is None:
        raise KeyError(f"No label column found in {cols}")

    drop = {label_col}
    if chunk_col:
        drop.add(chunk_col)
    for c in cols:
        if lower[c] in ("subject", "subject_id", "sub", "time", "timestamp", "index",
                        "unnamed: 0", "session", "condition"):
            drop.add(c)

    feat_cols = [c for c in cols
                 if c not in drop and pd.api.types.is_numeric_dtype(df[c])]
    return label_col, chunk_col, feat_cols


LABEL_COL, CHUNK_COL, FEAT_COLS = resolve_schema(_probe)
N_CHANNELS = len(FEAT_COLS)

print("label column  :", LABEL_COL)
print("chunk column  :", CHUNK_COL)
print("feature cols  :", N_CHANNELS, FEAT_COLS)
print("label values  :", sorted(_probe[LABEL_COL].dropna().unique().tolist()))

# Channel role tagging for the hemodynamic analyses.
# fNIRS2MW channel names follow <PAIR>_<I|PHI>_<O|DO>: intensity/phase, oxy/deoxy.
def tag_channel(name):
    n = name.upper()
    oxy = "DO" not in n.replace("_DO", "|DO").split("|")[-1] if "_DO" in n else True
    is_deoxy = bool(re.search(r"_DO$|_DEOXY|HHB|HBR", n))
    is_oxy   = (not is_deoxy) and bool(re.search(r"_O$|_OXY|HBO", n))
    is_phase = bool(re.search(r"PHI|PHASE", n))
    return {"deoxy": is_deoxy, "oxy": is_oxy, "phase": is_phase}

CH_TAGS  = {c: tag_channel(c) for c in FEAT_COLS}
OXY_IDX   = [i for i, c in enumerate(FEAT_COLS) if CH_TAGS[c]["oxy"] and not CH_TAGS[c]["phase"]]
DEOXY_IDX = [i for i, c in enumerate(FEAT_COLS) if CH_TAGS[c]["deoxy"] and not CH_TAGS[c]["phase"]]
if not OXY_IDX:
    OXY_IDX = list(range(0, max(1, N_CHANNELS // 2)))
if not DEOXY_IDX:
    DEOXY_IDX = list(range(max(1, N_CHANNELS // 2), N_CHANNELS))

print("oxy-like idx   :", OXY_IDX, [FEAT_COLS[i] for i in OXY_IDX])
print("deoxy-like idx :", DEOXY_IDX, [FEAT_COLS[i] for i in DEOXY_IDX])


In [ ]:
# ============================================================
# 06 | PER-SUBJECT CACHE BUILD (resumable, skips existing)
# ============================================================

def subject_id_from_path(p):
    stem = Path(p).stem
    m = re.search(r"(\d+)", stem)
    return f"sub_{int(m.group(1)):03d}" if m else stem


def build_subject_cache(csv_path, force=False):
    sid = subject_id_from_path(csv_path)
    out = Path(CFG.CACHE_DIR) / f"{sid}.npz"
    if out.exists() and not force:
        return sid, "skipped"

    df = pd.read_csv(csv_path)
    df = df.dropna(subset=[LABEL_COL])

    if CHUNK_COL is not None:
        # rows are timesteps; group by window id
        df = df.sort_values([CHUNK_COL])
        g = df.groupby(CHUNK_COL, sort=True)
        keys, Xs, ys = [], [], []
        for k, sub in g:
            arr = sub[FEAT_COLS].to_numpy(dtype=np.float32)
            if arr.shape[0] < CFG.N_TIMESTEPS:
                continue
            arr = arr[:CFG.N_TIMESTEPS]
            lab = sub[LABEL_COL].to_numpy()
            if len(np.unique(lab)) != 1:
                continue                      # discard windows straddling a condition change
            keys.append(k); Xs.append(arr); ys.append(int(lab[0]))
        if not Xs:
            return sid, "empty"
        X = np.stack(Xs).astype(np.float32)   # (N, T, C)
        y = np.asarray(ys, dtype=np.int64)
        chunk = np.asarray(keys)
    else:
        # already one row per window: reshape flat features into (T, C)
        y = df[LABEL_COL].to_numpy(dtype=np.int64)
        flat = df[FEAT_COLS].to_numpy(dtype=np.float32)
        C = flat.shape[1] // CFG.N_TIMESTEPS
        X = flat.reshape(len(df), CFG.N_TIMESTEPS, C)
        chunk = np.arange(len(df))

    # chunk order defines recording time; needed for the leakage guard
    order = np.argsort(chunk.astype(float) if np.issubdtype(chunk.dtype, np.number)
                       else np.arange(len(chunk)))
    X, y, chunk = X[order], y[order], chunk[order]

    np.savez_compressed(out, X=X, y=y, chunk=chunk,
                        channels=np.array(FEAT_COLS, dtype=object))
    return sid, f"cached {X.shape}"


t0 = time.time()
status = []
for i, p in enumerate(CSV_FILES):
    sid, msg = build_subject_cache(p)
    status.append((sid, msg))
    if CFG.VERBOSE and (i % 10 == 0 or "cached" in msg):
        print(f"[{i+1:3d}/{len(CSV_FILES)}] {sid}: {msg}")

n_new = sum(1 for _, m in status if m.startswith("cached"))
print(f"\ncache complete in {time.time()-t0:.1f}s | newly cached: {n_new} | total on disk: "
      f"{len(list(Path(CFG.CACHE_DIR).glob('*.npz')))}")


In [ ]:
# ============================================================
# 07 | LOAD CACHE INTO MEMORY, FILTER TO THE TASK LABELS
# ============================================================

def load_all_subjects(task_labels):
    lab_map = {int(l): i for i, l in enumerate(sorted(task_labels))}
    data = {}
    for p in sorted(Path(CFG.CACHE_DIR).glob("*.npz")):
        z = np.load(p, allow_pickle=True)
        X, y = z["X"], z["y"]
        m = np.isin(y, list(lab_map.keys()))
        if m.sum() < 40:
            continue
        Xf = X[m]
        yf = np.array([lab_map[int(v)] for v in y[m]], dtype=np.int64)
        if len(np.unique(yf)) < 2:
            continue                                   # unusable as a client
        data[p.stem] = {"X": Xf, "y": yf}
    return data


SUBJECTS = load_all_subjects(CFG.TASK_LABELS)
SUBJ_IDS = sorted(SUBJECTS.keys())
N_CH = SUBJECTS[SUBJ_IDS[0]]["X"].shape[2]

print(f"usable subjects : {len(SUBJ_IDS)}")
print(f"window shape    : (T={SUBJECTS[SUBJ_IDS[0]]['X'].shape[1]}, C={N_CH})")
print(f"total windows   : {sum(v['X'].shape[0] for v in SUBJECTS.values()):,}")
print(f"task labels     : {CFG.TASK_LABELS} -> classes 0..{CFG.N_CLASSES-1}")

meta = pd.DataFrame([
    {"subject": s,
     "n_windows": SUBJECTS[s]["X"].shape[0],
     **{f"n_class{c}": int((SUBJECTS[s]["y"] == c).sum()) for c in range(CFG.N_CLASSES)},
     "class_balance": float(np.bincount(SUBJECTS[s]["y"], minlength=CFG.N_CLASSES).min() /
                            max(1, np.bincount(SUBJECTS[s]["y"], minlength=CFG.N_CLASSES).max()))}
    for s in SUBJ_IDS
])
meta.to_csv(f"{CFG.TAB_DIR}/subject_meta.csv", index=False)
display(meta.describe().T[["mean", "std", "min", "max"]].round(2))


In [ ]:
# ============================================================
# 08 | FIGURE 1 - DATASET AND CLIENT OVERVIEW  (v2)
# ============================================================
# fNIRS2MW is volume- and label-balanced by protocol: every subject contributes
# exactly the same number of windows with an exactly even class split. The old
# panels (a) and (b) were therefore flat lines. Heterogeneity here is physiological,
# not statistical, so the figure now shows that instead.
print("windows per subject :", meta['n_windows'].unique())
print("class balance       :", meta['class_balance'].unique())

fig, axes = plt.subplots(1, 3, figsize=(COL2, 2.2), constrained_layout=True)

ax = axes[0]
amp = HET.set_index("subject")["hbo_amplitude"].reindex(SUBJ_IDS).to_numpy() \
      if "HET" in dir() else None
if amp is None or np.all(np.isnan(amp)):
    # HET is built in cell 10; fall back to a direct computation
    amp = []
    for s in SUBJ_IDS:
        X, y = SUBJECTS[s]["X"], SUBJECTS[s]["y"]
        b = X[:, :int(2 * 5.2), :].mean(axis=1, keepdims=True)
        o = (X - b)[:, :, OXY_IDX].mean(axis=2)
        amp.append(np.max(np.abs(o[y == 1].mean(0) - o[y == 0].mean(0))))
    amp = np.array(amp)
o = np.argsort(amp)
ax.bar(np.arange(len(amp)), amp[o], width=1.0, color=PAL["blue"], linewidth=0)
ax.set_xlim(-0.5, len(amp) - 0.5)
ax.set_xlabel("Subject (sorted)"); ax.set_ylabel("Peak class contrast")
ax.set_title("(a) Task-evoked response", loc="left")

ax = axes[1]
scale = np.array([SUBJECTS[s]["X"].reshape(-1, N_CH).std(axis=0).mean() for s in SUBJ_IDS])
ax.hist(scale, bins=24, color=PAL["purple"], edgecolor="white", linewidth=0.5)
ax.set_yscale("log")
ax.set_xlabel("Baseline signal SD (a.u.)"); ax.set_ylabel("Subjects (log)")
ax.set_title("(b) Optical coupling spread", loc="left")

ax = axes[2]
sub = np.concatenate([SUBJECTS[s]["X"][:300] for s in SUBJ_IDS[:40]], axis=0)
chan_std = sub.std(axis=(0, 1))
cols = [PAL["blue"] if i in OXY_IDX else (PAL["purple"] if i in DEOXY_IDX else PAL["gray"])
        for i in range(N_CH)]
ax.bar(np.arange(N_CH), chan_std, color=cols, linewidth=0)
ax.set_xticks(np.arange(N_CH))
ax.set_xticklabels([c.replace("_", "\n") for c in FEAT_COLS], fontsize=5.5)
ax.set_ylabel("Signal SD (a.u.)")
ax.set_title("(c) Channel scale", loc="left")
del sub

savefig(fig, "fig01_dataset_overview", tight=False)
print(f"\nsignal scale across clients: min={scale.min():.3f} max={scale.max():.3f} "
      f"ratio={scale.max()/scale.min():.1f}x")


In [ ]:
# ============================================================
# 09 | FIGURE 2 - GRAND-AVERAGE HEMODYNAMIC RESPONSE BY WORKLOAD CLASS
# ============================================================
FS = 5.2  # fNIRS2MW sampling rate, Hz -> 150 timesteps == ~28.8 s
tvec = np.arange(CFG.N_TIMESTEPS) / FS

def grand_average(idx_channels):
    out = {c: [] for c in range(CFG.N_CLASSES)}
    for s in SUBJ_IDS:
        X, y = SUBJECTS[s]["X"], SUBJECTS[s]["y"]
        # baseline-correct each window on its first 2 s, then average within subject
        base = X[:, :int(2 * FS), :].mean(axis=1, keepdims=True)
        Xb = (X - base)[:, :, idx_channels].mean(axis=2)
        for c in range(CFG.N_CLASSES):
            m = y == c
            if m.sum() > 5:
                out[c].append(Xb[m].mean(axis=0))
    return {c: np.stack(v) for c, v in out.items() if v}

ga_oxy   = grand_average(OXY_IDX)
ga_deoxy = grand_average(DEOXY_IDX)

fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.3), sharex=True)
for ax, ga, title in zip(axes, [ga_oxy, ga_deoxy],
                         ["(a) Oxy-like channels", "(b) Deoxy-like channels"]):
    for c in range(CFG.N_CLASSES):
        if c not in ga:
            continue
        m = ga[c].mean(axis=0)
        se = ga[c].std(axis=0) / np.sqrt(ga[c].shape[0])
        ax.plot(tvec, m, color=PAL_SEQ[c], label=f"{CFG.TASK_LABELS[c]}-back")
        ax.fill_between(tvec, m - 1.96 * se, m + 1.96 * se, color=PAL_SEQ[c], alpha=0.18, linewidth=0)
    ax.axhline(0, color=PAL["gray"], linewidth=0.6)
    ax.set_xlabel("Time within window (s)")
    ax.set_title(title, loc="left")
axes[0].set_ylabel("Baseline-corrected signal (a.u.)")
axes[0].legend(title="Workload")
savefig(fig, "fig02_grand_average_response")

# point-wise paired test between the two extreme workload levels
if CFG.N_CLASSES >= 2 and 0 in ga_oxy and (CFG.N_CLASSES - 1) in ga_oxy:
    a, b = ga_oxy[0], ga_oxy[CFG.N_CLASSES - 1]
    n = min(a.shape[0], b.shape[0])
    tstat, pvals = st.ttest_rel(a[:n], b[:n], axis=0)
    # Benjamini-Hochberg over timepoints
    ordp = np.argsort(pvals); ranked = pvals[ordp]
    thresh = 0.05 * (np.arange(1, len(ranked) + 1) / len(ranked))
    sig = np.zeros_like(pvals, dtype=bool)
    passed = ranked <= thresh
    if passed.any():
        sig[ordp[:np.max(np.where(passed)[0]) + 1]] = True
    print(f"Oxy grand-average: {sig.sum()}/{len(sig)} timepoints differ between "
          f"{CFG.TASK_LABELS[0]}-back and {CFG.TASK_LABELS[-1]}-back (paired t, BH q<0.05)")


In [ ]:
# ============================================================
# 10 | PER-CLIENT HETEROGENEITY DESCRIPTORS
# ============================================================

def js_divergence(p, q, eps=1e-12):
    p = np.asarray(p, float) + eps; p /= p.sum()
    q = np.asarray(q, float) + eps; q /= q.sum()
    m = 0.5 * (p + q)
    kl = lambda a, b: np.sum(a * np.log(a / b))
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


def client_descriptors(sid):
    X, y = SUBJECTS[sid]["X"], SUBJECTS[sid]["y"]
    base = X[:, :int(2 * FS), :].mean(axis=1, keepdims=True)
    Xb = X - base

    oxy = Xb[:, :, OXY_IDX].mean(axis=2)
    hi = oxy[y == (CFG.N_CLASSES - 1)]
    lo = oxy[y == 0]

    # contrast waveform between the highest and lowest workload condition
    if len(hi) > 3 and len(lo) > 3:
        contrast = hi.mean(axis=0) - lo.mean(axis=0)
        amp = float(np.max(np.abs(contrast)))
        lat = float(np.argmax(np.abs(contrast)) / FS)
        # Cohen's d at the peak timepoint
        tp = int(np.argmax(np.abs(contrast)))
        sp = np.sqrt(((len(hi) - 1) * hi[:, tp].var(ddof=1) +
                      (len(lo) - 1) * lo[:, tp].var(ddof=1)) / max(1, len(hi) + len(lo) - 2))
        d = float((hi[:, tp].mean() - lo[:, tp].mean()) / (sp + 1e-9))
    else:
        amp, lat, d = np.nan, np.nan, np.nan

    counts = np.bincount(y, minlength=CFG.N_CLASSES)
    uniform = np.ones(CFG.N_CLASSES) / CFG.N_CLASSES

    return {
        "subject":        sid,
        "n_windows":      int(len(y)),
        "label_js":       float(js_divergence(counts, uniform)),
        "hbo_amplitude":  amp,
        "hbo_latency_s":  lat,
        "peak_cohen_d":   d,
        "signal_scale":   float(X.reshape(-1, X.shape[2]).std(axis=0).mean()),
        "within_var":     float(Xb.var()),
        "snr_proxy":      float(np.nan if np.isnan(amp) else amp / (Xb.std() + 1e-9)),
    }


HET = pd.DataFrame([client_descriptors(s) for s in SUBJ_IDS])

# distance of each client's mean feature vector from the global mean (covariate shift)
gm = np.concatenate([SUBJECTS[s]["X"].reshape(-1, N_CH) for s in SUBJ_IDS], axis=0).mean(axis=0)
HET["mean_shift"] = [float(np.linalg.norm(SUBJECTS[s]["X"].reshape(-1, N_CH).mean(axis=0) - gm))
                     for s in SUBJ_IDS]

HET.to_csv(f"{CFG.TAB_DIR}/client_heterogeneity.csv", index=False)
display(HET.describe().T[["mean", "std", "min", "max"]].round(4))
print("\nCoefficient of variation across clients:")
for c in ["hbo_amplitude", "hbo_latency_s", "signal_scale", "mean_shift"]:
    v = HET[c].dropna()
    print(f"  {c:16s} CV = {v.std()/abs(v.mean()):.3f}")


In [ ]:
# ============================================================
# 11 | FIGURE 3 - NON-IID CHARACTERISATION
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(COL2, 2.0))

ax = axes[0]
ax.hist(HET["hbo_amplitude"].dropna(), bins=20, color=PAL["blue"], edgecolor="white", linewidth=0.5)
ax.set_xlabel("Peak contrast amplitude"); ax.set_ylabel("Clients")
ax.set_title("(a) Response amplitude", loc="left")

ax = axes[1]
ax.hist(HET["hbo_latency_s"].dropna(), bins=20, color=PAL["purple"], edgecolor="white", linewidth=0.5)
ax.set_xlabel("Latency to peak (s)")
ax.set_title("(b) Response latency", loc="left")

ax = axes[2]
ax.scatter(HET["signal_scale"], HET["hbo_amplitude"], s=14, color=PAL["teal"],
           edgecolor="white", linewidth=0.4)
ax.set_xlabel("Baseline signal scale"); ax.set_ylabel("Peak amplitude")
ax.set_title("(c) Scale vs amplitude", loc="left")

ax = axes[3]
ax.hist(HET["label_js"], bins=20, color=PAL["coral"], edgecolor="white", linewidth=0.5)
ax.set_xlabel("JS divergence from uniform")
ax.set_title("(d) Label skew", loc="left")

savefig(fig, "fig03_noniid_characterisation")

r, p = st.spearmanr(HET["signal_scale"], HET["hbo_amplitude"], nan_policy="omit")
print(f"Spearman(signal scale, peak amplitude) rho={r:.3f}, p={p:.2e}")


In [ ]:
# ============================================================
# 12 | SUBJECT-LEVEL AND WITHIN-SUBJECT TEMPORAL SPLITS  (v2)
# ============================================================

def subject_split(seed):
    # Plain shuffled split. The v1 volume stratification was a no-op here because
    # every subject holds exactly the same number of windows.
    rng = np.random.RandomState(seed)
    ids = np.array(SUBJ_IDS)
    perm = rng.permutation(len(ids))
    test  = sorted(ids[perm[:CFG.N_TEST_SUBJECTS]].tolist())
    val   = sorted(ids[perm[CFG.N_TEST_SUBJECTS:
                            CFG.N_TEST_SUBJECTS + CFG.N_VAL_SUBJECTS]].tolist())
    train = sorted(ids[perm[CFG.N_TEST_SUBJECTS + CFG.N_VAL_SUBJECTS:]].tolist())
    return train, val, test


def temporal_split(sid, train_frac=None, val_frac=None, guard=None):
    train_frac = CFG.TRAIN_FRAC if train_frac is None else train_frac
    val_frac   = CFG.VAL_FRAC if val_frac is None else val_frac
    guard      = CFG.GUARD_CHUNKS if guard is None else guard
    n = SUBJECTS[sid]["X"].shape[0]
    i_tr = int(n * train_frac)
    i_va = int(n * (train_frac + val_frac))
    tr = np.arange(0, max(0, i_tr - guard))
    va = np.arange(i_tr, max(i_tr, i_va - guard))
    te = np.arange(i_va, n)
    return tr, va, te


def get_xy(sid, idx, norm_stats=None):
    X = SUBJECTS[sid]["X"][idx].astype(np.float32)
    y = SUBJECTS[sid]["y"][idx]
    if norm_stats is not None:
        mu, sd = norm_stats
        X = (X - mu) / (sd + 1e-6)
    return np.transpose(X, (0, 2, 1)), y


def client_norm_stats(sid, tr_idx):
    X = SUBJECTS[sid]["X"][tr_idx]
    return X.reshape(-1, N_CH).mean(0).astype(np.float32), \
           X.reshape(-1, N_CH).std(0).astype(np.float32)


def make_loader(X, y, bs, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, drop_last=False,
                      num_workers=0, pin_memory=(DEVICE.type == "cuda"))


TRAIN_S, VAL_S, TEST_S = subject_split(CFG.SEED)
print(f"per replicate: {len(TRAIN_S)} train clients / {len(VAL_S)} val / {len(TEST_S)} test")

# how many distinct subjects will ever appear as a test target across replicates
seen = set()
for s in CFG.SEEDS:
    seen |= set(subject_split(s)[2])
print(f"distinct test subjects across {len(CFG.SEEDS)} replicates: {len(seen)}/{len(SUBJ_IDS)}")


In [ ]:
# ============================================================
# 13 | MODEL ZOO  (v2)
# ============================================================

class ClientAffine(nn.Module):
    # Per-channel scale and shift, held local under FedBN / FedPer / FedAff.
    # This is the parameter block that directly absorbs optode-coupling variation,
    # which is the heterogeneity actually present in this dataset. In v1 the only
    # personal parameters were BatchNorm (4% of the model) and the head (0.8%),
    # so FedBN and FedPer were structurally almost identical to FedAvg.
    def __init__(self, n_ch):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(1, n_ch, 1))
        self.shift = nn.Parameter(torch.zeros(1, n_ch, 1))

    def forward(self, x):
        return x * self.scale + self.shift


class FNIRSNet(nn.Module):
    def __init__(self, n_ch, n_cls, F1=None, D=None, F2=None,
                 k1=51, k2=17, p_drop=0.35, T=150):
        super().__init__()
        F1 = F1 or CFG.F1; D = D or CFG.D_MULT; F2 = F2 or CFG.F2
        self.client_affine = ClientAffine(n_ch)
        self.temporal = nn.Sequential(
            nn.Conv1d(n_ch, F1, k1, padding=k1 // 2, bias=False), nn.BatchNorm1d(F1))
        self.depthwise = nn.Sequential(
            nn.Conv1d(F1, F1 * D, 1, groups=F1, bias=False),
            nn.BatchNorm1d(F1 * D), nn.ELU(), nn.AvgPool1d(4), nn.Dropout(p_drop))
        self.separable = nn.Sequential(
            nn.Conv1d(F1 * D, F1 * D, k2, padding=k2 // 2, groups=F1 * D, bias=False),
            nn.Conv1d(F1 * D, F2, 1, bias=False),
            nn.BatchNorm1d(F2), nn.ELU(), nn.AvgPool1d(4), nn.Dropout(p_drop))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(F2, n_cls)

    def features(self, x):
        x = self.client_affine(x)
        x = self.temporal(x); x = self.depthwise(x); x = self.separable(x)
        return self.pool(x).flatten(1)

    def forward(self, x):
        return self.classifier(self.features(x))


class DeepConv1D(nn.Module):
    def __init__(self, n_ch, n_cls, p_drop=0.4, **kw):
        super().__init__()
        self.client_affine = ClientAffine(n_ch)
        def blk(i, o, k):
            return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2, bias=False),
                                 nn.BatchNorm1d(o), nn.ELU(),
                                 nn.MaxPool1d(3), nn.Dropout(p_drop))
        self.body = nn.Sequential(blk(n_ch, 32, 25), blk(32, 64, 11), blk(64, 128, 7))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(128, n_cls)

    def features(self, x):
        return self.pool(self.body(self.client_affine(x))).flatten(1)

    def forward(self, x):
        return self.classifier(self.features(x))


class GRUNet(nn.Module):
    def __init__(self, n_ch, n_cls, hid=64, p_drop=0.3, **kw):
        super().__init__()
        self.client_affine = ClientAffine(n_ch)
        self.norm = nn.BatchNorm1d(n_ch)
        self.rnn = nn.GRU(n_ch, hid, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=p_drop)
        self.classifier = nn.Linear(hid * 2, n_cls)

    def features(self, x):
        x = self.norm(self.client_affine(x)).transpose(1, 2)
        o, _ = self.rnn(x)
        return o.mean(dim=1)

    def forward(self, x):
        return self.classifier(self.features(x))


class TinyTransformer(nn.Module):
    def __init__(self, n_ch, n_cls, d=64, heads=4, layers=2, patch=10,
                 p_drop=0.2, T=150, **kw):
        super().__init__()
        self.client_affine = ClientAffine(n_ch)
        self.norm = nn.BatchNorm1d(n_ch)
        self.patch = nn.Conv1d(n_ch, d, patch, stride=patch)
        self.pos = nn.Parameter(torch.zeros(1, T // patch, d))
        enc = nn.TransformerEncoderLayer(d, heads, d * 2, p_drop,
                                         batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc, layers)
        self.classifier = nn.Linear(d, n_cls)

    def features(self, x):
        x = self.patch(self.norm(self.client_affine(x))).transpose(1, 2)
        x = x + self.pos[:, :x.size(1)]
        return self.enc(x).mean(dim=1)

    def forward(self, x):
        return self.classifier(self.features(x))


MODEL_REGISTRY = {"fnirsnet": FNIRSNet, "deepconv": DeepConv1D,
                  "gru": GRUNet, "tinytx": TinyTransformer}


def build_model(name=None):
    name = name or CFG.DEFAULT_MODEL
    return MODEL_REGISTRY[name](n_ch=N_CH, n_cls=CFG.N_CLASSES, T=CFG.N_TIMESTEPS).to(DEVICE)


for nm in CFG.MODEL_ZOO:
    m = build_model(nm)
    n_par = sum(p.numel() for p in m.parameters() if p.requires_grad)
    with torch.no_grad():
        out = m(torch.zeros(2, N_CH, CFG.N_TIMESTEPS, device=DEVICE))
    print(f"{nm:10s} params={n_par:>8,}  out={tuple(out.shape)}")
    del m
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# 14 | METRICS AND LOCAL TRAINING  (v2.1, NaN-safe)
# ============================================================

def make_scaler():
    try:
        return torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def compute_metrics(y_true, y_pred, y_prob=None):
    out = {
        "acc":   float(accuracy_score(y_true, y_pred)),
        "bacc":  float(balanced_accuracy_score(y_true, y_pred)),
        "f1":    float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
    }
    # A diverged model (high DP noise) produces non-finite logits. Score it as
    # chance rather than raising, so the utility cliff lands in the results.
    if y_prob is None or not np.isfinite(y_prob).all():
        out["auroc"] = float("nan")
        return out
    try:
        if CFG.N_CLASSES == 2 and len(np.unique(y_true)) == 2:
            out["auroc"] = float(roc_auc_score(y_true, y_prob[:, 1]))
        elif len(np.unique(y_true)) == CFG.N_CLASSES:
            out["auroc"] = float(roc_auc_score(y_true, y_prob, multi_class="ovr"))
        else:
            out["auroc"] = float("nan")
    except ValueError:
        out["auroc"] = float("nan")
    return out


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    P, Y = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            logits = model(xb)
        p = torch.softmax(logits.float(), dim=1).cpu().numpy()
        P.append(p); Y.append(yb.numpy())
    P = np.concatenate(P); Y = np.concatenate(Y)
    if not np.isfinite(P).all():
        # uniform posterior -> argmax is a constant predictor -> bAcc 0.5
        bad = ~np.isfinite(P).all(axis=1)
        P[bad] = 1.0 / CFG.N_CLASSES
    return compute_metrics(Y, P.argmax(1), P), P, Y


def local_train(model, loader, epochs, lr, global_params=None, mu=0.0, class_weight=None):
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG.WEIGHT_DECAY)
    scaler = make_scaler()
    crit = nn.CrossEntropyLoss(weight=class_weight)
    tot, nb = 0.0, 0
    for _ in range(epochs):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True); yb = yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=USE_AMP):
                loss = crit(model(xb), yb)
            if mu > 0 and global_params is not None:
                prox = 0.0
                for p, g in zip(model.parameters(), global_params):
                    prox = prox + ((p - g.to(p.device)) ** 2).sum()
                loss = loss + 0.5 * mu * prox
            if not torch.isfinite(loss):
                continue                      # skip the batch, do not poison the weights
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
            scaler.step(opt); scaler.update()
            tot += float(loss.detach()); nb += 1
    return tot / max(1, nb)

In [ ]:
# ============================================================
# 15 | AGGREGATION RULES  (v2)
# ============================================================

def get_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def norm_prefixes(model):
    pre = set()
    for name, mod in model.named_modules():
        if isinstance(mod, (nn.BatchNorm1d, nn.BatchNorm2d, nn.LayerNorm,
                            nn.GroupNorm, nn.InstanceNorm1d)):
            pre.add(name)
    return pre


NORM_PREFIXES = {m: norm_prefixes(MODEL_REGISTRY[m](n_ch=N_CH, n_cls=CFG.N_CLASSES,
                                                    T=CFG.N_TIMESTEPS))
                 for m in CFG.MODEL_ZOO}


def is_norm_key(k, model_name=None):
    pres = NORM_PREFIXES.get(model_name or CFG.DEFAULT_MODEL, set())
    if any(k == p or k.startswith(p + ".") for p in pres):
        return True
    return bool(re.search(r"running_mean|running_var|num_batches_tracked", k))


def is_head_key(k):
    return k.startswith("classifier")


def is_affine_key(k):
    return k.startswith("client_affine")


def personal_fn_for(method, model_name=None):
    # fedavg / fedprox aggregate everything, including the affine layer, so it
    # becomes a shared global rescaling and the comparison stays fair.
    if method in ("fedavg", "fedprox"):
        return lambda k: False
    if method == "fedaff":                       # isolates the affine contribution
        return is_affine_key
    if method == "fedbn":
        return lambda k: is_norm_key(k, model_name) or is_affine_key(k)
    if method == "fedper":
        return lambda k: is_head_key(k) or is_affine_key(k)
    raise ValueError(method)


PERSONAL_KEY_FN = {m: personal_fn_for(m)
                   for m in ["fedavg", "fedprox", "fedaff", "fedbn", "fedper"]}


def aggregate(global_state, client_states, weights, personal_fn, dp_sigma=0.0, rng=None):
    total = float(sum(weights))
    new = {k: v.clone() for k, v in global_state.items()}
    for k, gv in global_state.items():
        if personal_fn(k):
            continue
        if not torch.is_floating_point(gv):
            new[k] = torch.stack([cs[k].float() for cs in client_states]).mean(0).round().to(gv.dtype)
            continue
        acc = torch.zeros_like(gv, dtype=torch.float32)
        for cs, w in zip(client_states, weights):
            acc += cs[k].float() * (w / total)
        if dp_sigma > 0 and rng is not None and acc.numel() > 1:
            acc += torch.from_numpy(rng.normal(0, dp_sigma, size=tuple(acc.shape)).astype(np.float32))
        new[k] = acc.to(gv.dtype)
    return new


def merge_personal(client_states, personal_fn, reference):
    out = {}
    for k, gv in reference.items():
        if not personal_fn(k):
            continue
        out[k] = (torch.stack([cs[k].float() for cs in client_states]).mean(0).to(gv.dtype)
                  if torch.is_floating_point(gv) else reference[k].clone())
    return out


# how much of the model each rule actually keeps local
_m = build_model()
_sd = _m.state_dict()
_tot = sum(v.numel() for v in _sd.values() if torch.is_floating_point(v))
for meth, fn in PERSONAL_KEY_FN.items():
    loc = sum(v.numel() for k, v in _sd.items() if fn(k) and torch.is_floating_point(v))
    print(f"{meth:8s} personal params = {loc:>6,} / {_tot:,}  ({100*loc/_tot:5.2f}%)")
del _m, _sd


In [ ]:
# ============================================================
# 16 | FEDERATED SERVER LOOP  (v2, smoothed model selection)
# ============================================================

def prepare_client_cache(subject_ids, normalize=True):
    cache = {}
    for sid in subject_ids:
        tr, va, te = temporal_split(sid)
        if len(tr) < 20 or len(te) < 10:
            continue
        stats = client_norm_stats(sid, tr) if normalize else None
        Xtr, ytr = get_xy(sid, tr, stats)
        Xva, yva = get_xy(sid, va, stats) if len(va) else (Xtr[:0], ytr[:0])
        Xte, yte = get_xy(sid, te, stats)
        cache[sid] = {"train": (Xtr, ytr), "val": (Xva, yva), "test": (Xte, yte), "stats": stats}
    return cache


def run_federated(method, seed, model_name=None, rounds=None, local_epochs=None,
                  client_frac=None, mu=None, dp_sigma=0.0, normalize=True,
                  tag="", force=False):
    model_name   = model_name or CFG.DEFAULT_MODEL
    rounds       = rounds or CFG.ROUNDS
    local_epochs = local_epochs or CFG.LOCAL_EPOCHS
    client_frac  = client_frac if client_frac is not None else CFG.CLIENT_FRAC
    mu           = CFG.FEDPROX_MU if (mu is None and method == "fedprox") else (mu or 0.0)

    conf = dict(proto=CFG.PROTOCOL, kind="federated", method=method, seed=seed,
                model=model_name, rounds=rounds, local_epochs=local_epochs,
                client_frac=client_frac, mu=mu, dp_sigma=dp_sigma, normalize=normalize,
                lr=CFG.LR, bs=CFG.BATCH_SIZE, labels=CFG.TASK_LABELS,
                n_val=CFG.N_VAL_SUBJECTS, smooth=CFG.VAL_SMOOTH, tag=tag)
    exp = cfg_hash(conf)

    if has_result(exp) and not force:
        print(f"[skip] {method} seed={seed} {tag} -> cached")
        return load_result(exp)

    seed_everything(seed)
    tr_s, va_s, te_s = subject_split(seed)
    clients  = prepare_client_cache(tr_s, normalize)
    val_sub  = prepare_client_cache(va_s, normalize)
    test_sub = prepare_client_cache(te_s, normalize)
    cids = sorted(clients.keys())

    model = build_model(model_name)
    personal_fn = personal_fn_for(method, model_name)

    ck = load_ckpt(exp)
    if ck is not None:
        start_round = ck["round"] + 1
        global_state, client_personal = ck["global_state"], ck["client_personal"]
        history, best = ck["history"], ck["best"]
        rng = np.random.RandomState(ck["rng_seed"])
        print(f"[resume] {method} seed={seed} {tag} from round {start_round}")
    else:
        start_round = 0
        global_state = get_state(model)
        client_personal = {c: {k: v.clone() for k, v in global_state.items() if personal_fn(k)}
                           for c in cids}
        history = []
        best = {"round": -1, "val_bacc": -1, "state": None, "personal": None}
        rng = np.random.RandomState(seed)

    n_sel = max(1, int(round(client_frac * len(cids))))
    recent = deque([h["val_bacc"] for h in history[-CFG.VAL_SMOOTH:]], maxlen=CFG.VAL_SMOOTH)
    t_start = time.time()
    diverged, merged = False, {}
    for rd in range(start_round, rounds):
        sel = rng.choice(cids, size=n_sel, replace=False).tolist()
        states, weights, losses = [], [], []

        for c in sel:
            model.load_state_dict(global_state, strict=True)
            if client_personal[c]:
                model.load_state_dict(client_personal[c], strict=False)
            Xtr, ytr = clients[c]["train"]
            loader = make_loader(Xtr, ytr, CFG.BATCH_SIZE, True)
            gparams = [p.detach().clone().to(DEVICE) for p in model.parameters()] if mu > 0 else None
            cnt = np.bincount(ytr, minlength=CFG.N_CLASSES).astype(np.float32)
            cw = torch.tensor(cnt.sum() / (CFG.N_CLASSES * np.maximum(cnt, 1)),
                              dtype=torch.float32, device=DEVICE)
            l = local_train(model, loader, local_epochs, CFG.LR, gparams, mu, cw)
            cs = get_state(model)
            states.append(cs); weights.append(len(ytr)); losses.append(l)
            client_personal[c] = {k: v.clone() for k, v in cs.items() if personal_fn(k)}
        
        global_state = aggregate(global_state, states, weights, personal_fn, dp_sigma, rng)
        if not all(torch.isfinite(v).all() for v in global_state.values()
                   if torch.is_floating_point(v)):
            print(f"  [diverged] non-finite parameters at round {rd} "
                  f"(dp_sigma={dp_sigma}); recording collapse to chance")
            diverged = True
            break

        merged = merge_personal([client_personal[c] for c in sel], personal_fn, global_state)
        val_scores = []
        for s, d in val_sub.items():
            model.load_state_dict(global_state, strict=True)
            if merged:
                model.load_state_dict(merged, strict=False)
            m, _, _ = evaluate(model, make_loader(*d["test"], 256, False))
            val_scores.append(m["bacc"])
        val_bacc = float(np.mean(val_scores)) if val_scores else float("nan")

        # ---- smoothed model selection ----
        # v1 selected on the raw per-round val score, which peaked at round 2-4 on
        # noise and then never recovered. Selection now uses a moving average and
        # only becomes eligible once the window is full.
        recent.append(val_bacc)
        val_smooth = float(np.mean(recent))
        eligible = len(recent) == CFG.VAL_SMOOTH

        history.append({"round": rd, "train_loss": float(np.mean(losses)),
                        "val_bacc": val_bacc, "val_smooth": val_smooth})

        if eligible and val_smooth > best["val_bacc"]:
            best = {"round": rd, "val_bacc": val_smooth,
                    "state": {k: v.clone() for k, v in global_state.items()},
                    "personal": {k: v.clone() for k, v in merged.items()}}

        if CFG.VERBOSE and (rd % 10 == 0 or rd == rounds - 1):
            print(f"  r{rd:03d} loss={np.mean(losses):.4f} val={val_bacc:.4f} "
                  f"smooth={val_smooth:.4f} best={best['val_bacc']:.4f}@r{best['round']} "
                  f"[{time.time()-t_start:.0f}s]")

        if (rd + 1) % CFG.CKPT_EVERY == 0 or rd == rounds - 1:
            save_ckpt(exp, {"round": rd, "global_state": global_state,
                            "client_personal": client_personal, "history": history,
                            "best": best, "rng_seed": int(rng.randint(0, 2**31 - 1))})

        if best["round"] >= 0 and rd - best["round"] >= CFG.EARLY_PATIENCE:
            print(f"  early stop at round {rd}")
            break
    
    if diverged or best["state"] is None:
        best = {"round": rd, "val_bacc": float("nan"),
                "state": global_state, "personal": {}}
    per_subject = {}
    for s, d in test_sub.items():
        model.load_state_dict(best["state"], strict=True)
        if best["personal"]:
            model.load_state_dict(best["personal"], strict=False)
        m, _, _ = evaluate(model, make_loader(*d["test"], 256, False))
        per_subject[s] = m

    record = {"exp": exp, "config": conf, "history": history,
              "best_round": best["round"], "best_val_bacc": best["val_bacc"],
              "per_subject": per_subject,
              "mean": {k: float(np.nanmean([v[k] for v in per_subject.values()]))
                       for k in ["acc", "bacc", "f1", "kappa", "auroc"]},
              "std": {k: float(np.nanstd([v[k] for v in per_subject.values()]))
                      for k in ["acc", "bacc", "f1", "kappa", "auroc"]},
              "runtime_s": time.time() - t_start,
              "n_clients": len(cids), "n_test": len(test_sub),
              "diverged": bool(diverged)}
    save_result(exp, record)
    torch.save({"global_state": best["state"], "personal": best["personal"], "config": conf},
               f"{CFG.CKPT_DIR}/{exp}_best.pt")
    clear_ckpt(exp)
    print(f"[done] {method} seed={seed} {tag} bacc={record['mean']['bacc']:.4f} "
          f"+-{record['std']['bacc']:.4f} val={best['val_bacc']:.4f} "
          f"gap={best['val_bacc']-record['mean']['bacc']:+.4f} ({record['runtime_s']:.0f}s)")
    return record


In [ ]:
# ============================================================
# 17 | BASELINES  (v2, protocol-tagged)
# ============================================================

def run_centralized(seed, model_name=None, epochs=40, normalize=True, tag="", force=False):
    model_name = model_name or CFG.DEFAULT_MODEL
    conf = dict(proto=CFG.PROTOCOL, kind="centralized", method="centralized", seed=seed,
                model=model_name, epochs=epochs, normalize=normalize, lr=CFG.LR,
                bs=CFG.BATCH_SIZE, labels=CFG.TASK_LABELS,
                n_val=CFG.N_VAL_SUBJECTS, smooth=CFG.VAL_SMOOTH, tag=tag)
    exp = cfg_hash(conf)
    if has_result(exp) and not force:
        print(f"[skip] centralized seed={seed} -> cached")
        return load_result(exp)

    seed_everything(seed)
    tr_s, va_s, te_s = subject_split(seed)
    clients  = prepare_client_cache(tr_s, normalize)
    val_sub  = prepare_client_cache(va_s, normalize)
    test_sub = prepare_client_cache(te_s, normalize)

    Xtr = np.concatenate([d["train"][0] for d in clients.values()])
    ytr = np.concatenate([d["train"][1] for d in clients.values()])
    loader = make_loader(Xtr, ytr, CFG.BATCH_SIZE, True)

    model = build_model(model_name)
    ck = load_ckpt(exp)
    start_ep, best = 0, {"ep": -1, "val": -1, "state": None}
    recent = deque(maxlen=CFG.VAL_SMOOTH)
    if ck:
        model.load_state_dict(ck["state"]); start_ep = ck["epoch"] + 1; best = ck["best"]
        recent = deque(ck.get("recent", []), maxlen=CFG.VAL_SMOOTH)
        print(f"[resume] centralized from epoch {start_ep}")

    cnt = np.bincount(ytr, minlength=CFG.N_CLASSES).astype(np.float32)
    cw = torch.tensor(cnt.sum() / (CFG.N_CLASSES * np.maximum(cnt, 1)),
                      dtype=torch.float32, device=DEVICE)
    t0 = time.time()
    for ep in range(start_ep, epochs):
        local_train(model, loader, 1, CFG.LR, None, 0.0, cw)
        v = float(np.mean([evaluate(model, make_loader(*d["test"], 256, False))[0]["bacc"]
                           for d in val_sub.values()]))
        recent.append(v)
        vs = float(np.mean(recent))
        if len(recent) == CFG.VAL_SMOOTH and vs > best["val"]:
            best = {"ep": ep, "val": vs, "state": get_state(model)}
        if ep % 10 == 0:
            print(f"  ep{ep:03d} val={v:.4f} smooth={vs:.4f} best={best['val']:.4f}")
        if (ep + 1) % CFG.CKPT_EVERY == 0:
            save_ckpt(exp, {"epoch": ep, "state": get_state(model), "best": best,
                            "recent": list(recent)})

    model.load_state_dict(best["state"])
    per_subject = {s: evaluate(model, make_loader(*d["test"], 256, False))[0]
                   for s, d in test_sub.items()}
    record = {"exp": exp, "config": conf, "per_subject": per_subject,
              "mean": {k: float(np.nanmean([v[k] for v in per_subject.values()]))
                       for k in ["acc", "bacc", "f1", "kappa", "auroc"]},
              "std": {k: float(np.nanstd([v[k] for v in per_subject.values()]))
                      for k in ["acc", "bacc", "f1", "kappa", "auroc"]},
              "runtime_s": time.time() - t0, "history": []}
    save_result(exp, record)
    torch.save({"global_state": best["state"], "personal": {}, "config": conf},
               f"{CFG.CKPT_DIR}/{exp}_best.pt")
    clear_ckpt(exp)
    print(f"[done] centralized bacc={record['mean']['bacc']:.4f}")
    return record


def run_local_only(seed, model_name=None, epochs=30, normalize=True, force=False):
    model_name = model_name or CFG.DEFAULT_MODEL
    conf = dict(proto=CFG.PROTOCOL, kind="local", method="local-only", seed=seed,
                model=model_name, epochs=epochs, normalize=normalize, lr=CFG.LR,
                labels=CFG.TASK_LABELS)
    exp = cfg_hash(conf)
    if has_result(exp) and not force:
        print("[skip] local-only -> cached")
        return load_result(exp)

    seed_everything(seed)
    _, _, te_s = subject_split(seed)
    test_sub = prepare_client_cache(te_s, normalize)
    per_subject, t0 = {}, time.time()
    for s, d in test_sub.items():
        model = build_model(model_name)
        local_train(model, make_loader(*d["train"], CFG.BATCH_SIZE, True), epochs, CFG.LR)
        per_subject[s] = evaluate(model, make_loader(*d["test"], 256, False))[0]
        del model
    record = {"exp": exp, "config": conf, "per_subject": per_subject,
              "mean": {k: float(np.nanmean([v[k] for v in per_subject.values()]))
                       for k in ["acc", "bacc", "f1", "kappa", "auroc"]},
              "std": {k: float(np.nanstd([v[k] for v in per_subject.values()]))
                      for k in ["acc", "bacc", "f1", "kappa", "auroc"]},
              "runtime_s": time.time() - t0, "history": []}
    save_result(exp, record)
    print(f"[done] local-only bacc={record['mean']['bacc']:.4f}")
    return record


def window_features(X):
    mean = X.mean(2); std = X.std(2)
    slope = (X[:, :, -1] - X[:, :, 0]) / X.shape[2]
    p2p = X.max(2) - X.min(2)
    sk = st.skew(X, axis=2); ku = st.kurtosis(X, axis=2)
    half = X.shape[2] // 2
    dmean = X[:, :, half:].mean(2) - X[:, :, :half].mean(2)
    auc = X.sum(2) / X.shape[2]
    return np.concatenate([mean, std, slope, p2p, sk, ku, dmean, auc], axis=1)


def run_classical(seed, kind="lr", normalize=True, force=False):
    conf = dict(proto=CFG.PROTOCOL, kind="classical", method=f"{kind}-feat", seed=seed,
                normalize=normalize, labels=CFG.TASK_LABELS)
    exp = cfg_hash(conf)
    if has_result(exp) and not force:
        print(f"[skip] {kind}-feat -> cached")
        return load_result(exp)

    seed_everything(seed)
    tr_s, _, te_s = subject_split(seed)
    clients  = prepare_client_cache(tr_s, normalize)
    test_sub = prepare_client_cache(te_s, normalize)
    Xtr = window_features(np.concatenate([d["train"][0] for d in clients.values()]))
    ytr = np.concatenate([d["train"][1] for d in clients.values()])
    sc = StandardScaler().fit(Xtr)
    clf = (LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0) if kind == "lr"
           else RandomForestClassifier(n_estimators=400, class_weight="balanced",
                                       n_jobs=-1, random_state=seed, min_samples_leaf=3))
    t0 = time.time(); clf.fit(sc.transform(Xtr), ytr)
    per_subject = {}
    for s, d in test_sub.items():
        prob = clf.predict_proba(sc.transform(window_features(d["test"][0])))
        per_subject[s] = compute_metrics(d["test"][1], prob.argmax(1), prob)
    record = {"exp": exp, "config": conf, "per_subject": per_subject,
              "mean": {k: float(np.nanmean([v[k] for v in per_subject.values()]))
                       for k in ["acc", "bacc", "f1", "kappa", "auroc"]},
              "std": {k: float(np.nanstd([v[k] for v in per_subject.values()]))
                      for k in ["acc", "bacc", "f1", "kappa", "auroc"]},
              "runtime_s": time.time() - t0, "history": []}
    save_result(exp, record)
    print(f"[done] {kind}-feat bacc={record['mean']['bacc']:.4f}")
    return record


In [ ]:
# ============================================================
# 18 | RUN THE MAIN COMPARISON  (v2, 12 replicates)
# ============================================================
MAIN_METHODS = ["fedavg", "fedprox", "fedaff", "fedbn", "fedper"]

t_all = time.time()
for i, seed in enumerate(CFG.SEEDS):
    print(f"\n{'='*60}\nREPLICATE {i+1}/{len(CFG.SEEDS)}  (seed {seed})\n{'='*60}")
    run_local_only(seed)
    run_classical(seed, "lr")
    run_classical(seed, "rf")
    run_centralized(seed)
    for meth in MAIN_METHODS:
        run_federated(meth, seed)
    print(f"  elapsed {(time.time()-t_all)/60:.1f} min")

print("\nresults on disk:", len(list(Path(CFG.RESULT_DIR).glob('*.json'))))


In [ ]:
# ============================================================
# 19 | CALIBRATION-BUDGET EXPERIMENT  (v2, contiguous + balanced block)
# ============================================================

def load_best_bundle(method, seed, **kw):
    conf = dict(proto=CFG.PROTOCOL, kind="federated", method=method, seed=seed,
                model=CFG.DEFAULT_MODEL, rounds=CFG.ROUNDS,
                local_epochs=CFG.LOCAL_EPOCHS, client_frac=CFG.CLIENT_FRAC,
                mu=(CFG.FEDPROX_MU if method == "fedprox" else 0.0),
                dp_sigma=0.0, normalize=True, lr=CFG.LR, bs=CFG.BATCH_SIZE,
                labels=CFG.TASK_LABELS, n_val=CFG.N_VAL_SUBJECTS,
                smooth=CFG.VAL_SMOOTH, tag="")
    conf.update(kw)
    p = Path(CFG.CKPT_DIR) / f"{cfg_hash(conf)}_best.pt"
    return torch.load(p, map_location="cpu") if p.exists() else None


def calibration_subset(Xtr, ytr, k, n_cls):
    # v1 took Xtr[-k:] and fell back to scattered class-balanced indices whenever
    # that tail was single-class, so different budgets sampled different
    # distributions and the curve was non-monotonic. This walks backwards from the
    # end of the training block and stops as soon as k/n_cls of each class is held,
    # giving a contiguous, class-balanced calibration session of fixed composition.
    per = max(1, k // n_cls)
    taken = {c: [] for c in range(n_cls)}
    for i in range(len(ytr) - 1, -1, -1):
        c = int(ytr[i])
        if c in taken and len(taken[c]) < per:
            taken[c].append(i)
        if all(len(v) >= per for v in taken.values()):
            break
    idx = np.sort(np.concatenate([np.asarray(v, dtype=int) for v in taken.values() if v]))
    return Xtr[idx], ytr[idx]


def run_calibration(method, seed, budgets=None, head_only=False, force=False):
    budgets = budgets or CFG.CALIB_BUDGETS
    conf = dict(proto=CFG.PROTOCOL, kind="calibration", method=method, seed=seed,
                model=CFG.DEFAULT_MODEL, budgets=budgets, head_only=head_only,
                epochs=CFG.CALIB_EPOCHS, lr=CFG.CALIB_LR, labels=CFG.TASK_LABELS)
    exp = cfg_hash(conf)
    if has_result(exp) and not force:
        print(f"[skip] calibration {method} seed={seed} -> cached")
        return load_result(exp)

    bundle = load_best_bundle(method, seed)
    if bundle is None:
        print(f"[warn] no bundle for {method} seed={seed}; run the main comparison first")
        return None

    seed_everything(seed)
    _, _, te_s = subject_split(seed)
    test_sub = prepare_client_cache(te_s, True)

    rows, t0 = [], time.time()
    for k in budgets:
        for s, d in test_sub.items():
            model = build_model()
            model.load_state_dict(bundle["global_state"], strict=True)
            if bundle["personal"]:
                model.load_state_dict(bundle["personal"], strict=False)
            if k > 0:
                Xk, yk = calibration_subset(*d["train"], k, CFG.N_CLASSES)
                if head_only:
                    for n_, p_ in model.named_parameters():
                        p_.requires_grad = is_head_key(n_) or is_affine_key(n_)
                local_train(model, make_loader(Xk, yk, min(32, len(yk)), True),
                            CFG.CALIB_EPOCHS, CFG.CALIB_LR)
            m, _, _ = evaluate(model, make_loader(*d["test"], 256, False))
            rows.append({"budget": k, "subject": s, "n_calib": int(k), **m})
            del model

    df = pd.DataFrame(rows)
    record = {"exp": exp, "config": conf, "rows": rows,
              "summary": df.groupby("budget")["bacc"].agg(["mean", "std"]).reset_index()
                           .to_dict("records"),
              "runtime_s": time.time() - t0}
    save_result(exp, record)
    print(f"[done] calibration {method} seed={seed}")
    return record


CALIB_SEEDS = CFG.SEEDS[:6]          # 6 replicates is enough for the curve
CALIB_RECORDS = []
for seed in CALIB_SEEDS:
    for meth in ["fedavg", "fedbn", "fedper"]:
        r = run_calibration(meth, seed)
        if r: CALIB_RECORDS.append(r)
    r = run_calibration("fedper", seed, head_only=True)
    if r: CALIB_RECORDS.append(r)
print("calibration records:", len(CALIB_RECORDS))


In [ ]:
# ============================================================
# 20 | ABLATIONS  (v2, replicated + widened DP sweep)
# ============================================================
ABL_SEEDS = CFG.SEEDS[:CFG.ABL_REPS]

if CFG.RUN_ABLATIONS:
    print("\n--- A1: local epochs ---")
    for le in [1, 2, 5]:
        for sd in ABL_SEEDS:
            run_federated("fedbn", sd, local_epochs=le, tag=f"abl_le{le}")

    print("\n--- A2: client participation ---")
    for cf in [0.2, 0.4, 0.8]:
        for sd in ABL_SEEDS:
            run_federated("fedbn", sd, client_frac=cf, tag=f"abl_cf{cf}")

    print("\n--- A3: FedProx mu ---")
    for mu in [0.001, 0.01, 0.1]:
        for sd in ABL_SEEDS:
            run_federated("fedprox", sd, mu=mu, tag=f"abl_mu{mu}")

    print("\n--- A4: per-client normalisation ---")
    for nz in [True, False]:
        for sd in ABL_SEEDS:
            run_federated("fedavg", sd, normalize=nz, tag=f"abl_norm{int(nz)}")

if CFG.RUN_DP:
    print("\n--- A5: noise on aggregated updates ---")
    for sg in CFG.DP_SIGMAS:
        for sd in ABL_SEEDS:
            run_federated("fedbn", sd, dp_sigma=sg, tag=f"abl_dp{sg}")

if CFG.RUN_MODEL_ZOO:
    print("\n--- A6: architecture ---")
    for nm in CFG.MODEL_ZOO:
        for sd in ABL_SEEDS:
            run_federated("fedbn", sd, model_name=nm, tag=f"abl_arch_{nm}")

print("\ntotal results on disk:", len(list(Path(CFG.RESULT_DIR).glob('*.json'))))


In [ ]:
# ============================================================
# 21 | AGGREGATE ALL RESULTS  (v2, subject-level pivot)
# ============================================================
PRETTY = {"local-only": "Local-only", "lr-feat": "LR-feat", "rf-feat": "RF-feat",
          "centralized": "Centralized", "fedavg": "FedAvg", "fedprox": "FedProx",
          "fedaff": "FedAff", "fedbn": "FedBN", "fedper": "FedPer"}
ORDER = ["Local-only", "LR-feat", "RF-feat", "FedAvg", "FedProx", "FedAff",
         "FedBN", "FedPer", "Centralized"]
METHOD_COLOR.setdefault("FedAff", PAL["ochre"])
METHOD_COLOR["Centralized"] = "#8C6D3F"

recs = all_results()

rows = []
for r in recs:
    c = r.get("config", {})
    if c.get("proto") != CFG.PROTOCOL:
        continue                                   # ignore v1 records
    if c.get("kind") == "calibration" or c.get("tag", "").startswith("abl"):
        continue
    if "per_subject" not in r:
        continue
    for s, m in r["per_subject"].items():
        rows.append({"method": PRETTY.get(c.get("method"), c.get("method")),
                     "seed": c.get("seed"), "subject": s, **m})
PS = pd.DataFrame(rows)
PS = PS[PS["method"].isin(ORDER)]
PS.to_csv(f"{CFG.TAB_DIR}/per_subject_results.csv", index=False)
print(f"observations: {len(PS)} rows | "
      f"{PS['subject'].nunique()} distinct subjects | {PS['seed'].nunique()} replicates")

# replicate-level view, kept for the convergence and per-replicate reporting
PIV_OBS = PS.pivot_table(index=["seed", "subject"], columns="method", values="bacc")

# subject-level view: average each subject's score over the replicates in which it
# was a target. This is the unit used for all inferential tests, because the same
# subject reappearing across replicates is not an independent observation.
piv = PS.pivot_table(index="subject", columns="method", values="bacc").dropna(how="any")
methods_present = [m for m in ORDER if m in piv.columns]
print(f"paired testing units: {piv.shape[0]} independent subjects x "
      f"{len(methods_present)} methods")

main = (PS.groupby("method")
          .agg(bacc_m=("bacc", "mean"), bacc_s=("bacc", "std"),
               f1_m=("f1", "mean"), f1_s=("f1", "std"),
               auroc_m=("auroc", "mean"), auroc_s=("auroc", "std"),
               kappa_m=("kappa", "mean"), kappa_s=("kappa", "std"))
          .reindex(methods_present).reset_index())

fmt = lambda m, s: f"{m:.3f} ({s:.3f})"
tab = pd.DataFrame({
    "Method":        main["method"],
    "Balanced Acc.": [fmt(a, b) for a, b in zip(main.bacc_m, main.bacc_s)],
    "Macro F1":      [fmt(a, b) for a, b in zip(main.f1_m, main.f1_s)],
    "AUROC":         [fmt(a, b) for a, b in zip(main.auroc_m, main.auroc_s)],
    "Cohen kappa":   [fmt(a, b) for a, b in zip(main.kappa_m, main.kappa_s)],
})
display(tab)
to_latex(tab, "tab01_main_comparison",
         f"Calibration-free workload decoding on held-out target subjects, "
         f"{len(CFG.SEEDS)} leave-14-subjects-out replicates. Mean (SD) over "
         f"subject-replicate observations.", "tab:main")

# validation-to-test gap, the v1 failure mode
gaps = [(PRETTY.get(r["config"]["method"], "?"), r["config"]["seed"],
         r.get("best_val_bacc", np.nan), r["mean"]["bacc"])
        for r in recs
        if r.get("config", {}).get("proto") == CFG.PROTOCOL
        and r.get("config", {}).get("kind") == "federated"
        and not r["config"].get("tag")]
if gaps:
    G = pd.DataFrame(gaps, columns=["method", "seed", "val", "test"])
    G["gap"] = G["val"] - G["test"]
    print(f"\nval-to-test gap: mean {G['gap'].mean():+.3f}, max {G['gap'].max():+.3f} "
          f"(v1 was around +0.29)")


In [ ]:
# ============================================================
# 22 | FIGURE 4 - FEDERATED CONVERGENCE
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.4))
for r in recs:
    c = r.get("config", {})
    if c.get("kind") != "federated" or c.get("tag") or c.get("seed") != CFG.SEEDS[0]:
        continue
    h = pd.DataFrame(r["history"])
    name = PRETTY.get(c["method"], c["method"])
    axes[0].plot(h["round"], h["train_loss"], color=METHOD_COLOR.get(name, PAL["gray"]), label=name)
    axes[1].plot(h["round"], h["val_bacc"], color=METHOD_COLOR.get(name, PAL["gray"]), label=name)

axes[0].set_xlabel("Communication round"); axes[0].set_ylabel("Mean local loss")
axes[0].set_title("(a) Client objective", loc="left")
axes[1].set_xlabel("Communication round"); axes[1].set_ylabel("Val balanced accuracy")
axes[1].axhline(1 / CFG.N_CLASSES, color=PAL["gray"], linestyle=":", linewidth=0.9)
axes[1].set_title("(b) Held-out subject generalisation", loc="left")
axes[1].legend(ncol=2)
savefig(fig, "fig04_convergence")


In [ ]:
# ============================================================
# 23 | STATISTICAL TESTING  (v2, independent subjects)
# ============================================================

def cliffs_delta(a, b):
    a, b = np.asarray(a), np.asarray(b)
    gt = sum((x > y) for x in a for y in b)
    lt = sum((x < y) for x in a for y in b)
    return (gt - lt) / (len(a) * len(b))


def cliff_label(d):
    ad = abs(d)
    return "negligible" if ad < 0.147 else "small" if ad < 0.33 else \
           "medium" if ad < 0.474 else "large"


def bootstrap_ci(x, n=10000, alpha=0.05, seed=0):
    rng = np.random.RandomState(seed)
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    if len(x) < 2:
        return float("nan"), float("nan")
    bs = rng.choice(x, size=(n, len(x)), replace=True).mean(axis=1)
    return float(np.percentile(bs, 100 * alpha / 2)), float(np.percentile(bs, 100 * (1 - alpha / 2)))


def holm(pvals, names):
    order = np.argsort(pvals); m = len(pvals)
    adj = np.empty(m); run = 0.0
    for i, o in enumerate(order):
        run = max(run, (m - i) * pvals[o])
        adj[o] = min(1.0, run)
    return {names[i]: float(adj[i]) for i in range(m)}


# `piv` is one row per subject (cell 21). v1 tested 42 subject-by-seed rows in which
# the same subject recurred across seeds, which violates the independence assumption
# of every test below and inflated the gender result in the fairness audit.
print(f"N = {piv.shape[0]} independent subjects")

REF = "FedBN" if "FedBN" in methods_present else methods_present[-1]
pairs = []
for m in methods_present:
    if m == REF:
        continue
    a, b = piv[REF].to_numpy(), piv[m].to_numpy()
    try:
        _, p = st.wilcoxon(a, b, alternative="two-sided", zero_method="wilcox")
    except ValueError:
        p = 1.0
    pairs.append((m, float(np.mean(a - b)), cliffs_delta(a, b), float(p)))

adj = holm([x[3] for x in pairs], [x[0] for x in pairs])
stat_rows = []
for m, diff, d, p in pairs:
    lo, hi = bootstrap_ci(piv[REF].to_numpy() - piv[m].to_numpy())
    stat_rows.append({"Comparison": f"{REF} vs {m}", "Delta bAcc": round(diff, 4),
                      "95% CI": f"[{lo:.3f}, {hi:.3f}]", "Cliff d": round(d, 3),
                      "Effect": cliff_label(d), "p (Wilcoxon)": f"{p:.2e}",
                      "p (Holm)": f"{adj[m]:.2e}",
                      "Sig.": "yes" if adj[m] < 0.05 else "no"})
STAT = pd.DataFrame(stat_rows)
display(STAT)
to_latex(STAT, "tab02_significance",
         f"Paired Wilcoxon signed-rank tests against {REF} across "
         f"{piv.shape[0]} independent target subjects, Holm-corrected. "
         "Effect sizes are Cliff's delta.", "tab:sig", float_fmt="%.4f")

fr_stat, fr_p = st.friedmanchisquare(*[piv[m].to_numpy() for m in methods_present])
print(f"\nFriedman chi2 = {fr_stat:.3f}, p = {fr_p:.3e}, "
      f"k={len(methods_present)}, N={piv.shape[0]}")

# is the best method above chance at all?
best_m = piv.mean().idxmax()
v = piv[best_m].to_numpy()
lo, hi = bootstrap_ci(v)
_, p_chance = st.wilcoxon(v - 1 / CFG.N_CLASSES)
print(f"\nbest method {best_m}: bAcc {v.mean():.4f}, 95% CI [{lo:.3f}, {hi:.3f}], "
      f"vs chance p={p_chance:.2e}")

# how bimodal is per-subject performance
above = (v > 0.6).sum(); below = (v < 0.55).sum()
print(f"subject decodability: {above}/{len(v)} above 0.60, {below}/{len(v)} below 0.55")


In [ ]:
# ============================================================
# 24 | FIGURE 5 - PER-SUBJECT DISTRIBUTION WITH SIGNIFICANCE
# ============================================================
fig, ax = plt.subplots(figsize=(COL2, 2.8))
data = [piv[m].to_numpy() for m in methods_present]
bp = ax.boxplot(data, positions=np.arange(len(methods_present)), widths=0.55,
                patch_artist=True, showfliers=False,
                medianprops=dict(color=PAL["dark"], linewidth=1.1),
                whiskerprops=dict(linewidth=0.8), capprops=dict(linewidth=0.8))
for patch, m in zip(bp["boxes"], methods_present):
    patch.set_facecolor(METHOD_COLOR.get(m, PAL["gray"]))
    patch.set_alpha(0.35); patch.set_linewidth(0.8)
    patch.set_edgecolor(METHOD_COLOR.get(m, PAL["gray"]))

rng = np.random.RandomState(0)
for i, m in enumerate(methods_present):
    v = piv[m].to_numpy()
    ax.scatter(i + rng.uniform(-0.14, 0.14, len(v)), v, s=7,
               color=METHOD_COLOR.get(m, PAL["gray"]), alpha=0.65,
               edgecolor="white", linewidth=0.3, zorder=3)

ax.axhline(1 / CFG.N_CLASSES, color=PAL["gray"], linestyle=":", linewidth=0.9)
ax.set_xticks(np.arange(len(methods_present)))
ax.set_xticklabels(methods_present, rotation=18, ha="right")
ax.set_ylabel("Balanced accuracy (target subject)")

# significance brackets vs the reference method
ref_i = methods_present.index(REF)
top = max(np.max(v) for v in data)
step = 0.035 * (top - min(np.min(v) for v in data) + 1e-6)
lvl = 0
for m, _, _, _ in pairs:
    if adj[m] >= 0.05:
        continue
    j = methods_present.index(m)
    yy = top + step * (lvl + 1)
    ax.plot([ref_i, ref_i, j, j], [yy - step * 0.25, yy, yy, yy - step * 0.25],
            color=PAL["dark"], linewidth=0.7)
    star = "***" if adj[m] < 1e-3 else "**" if adj[m] < 1e-2 else "*"
    ax.text((ref_i + j) / 2, yy, star, ha="center", va="bottom", fontsize=8)
    lvl += 1
savefig(fig, "fig05_per_subject_boxplot")


In [ ]:
# ============================================================
# 25 | FIGURE 6 - CRITICAL DIFFERENCE DIAGRAM (Friedman + Nemenyi)
# ============================================================
Q005 = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949,
        8: 3.031, 9: 3.102, 10: 3.164}

ranks = piv[methods_present].rank(axis=1, ascending=False).mean(axis=0)
k, N = len(methods_present), piv.shape[0]
CD = Q005.get(k, 3.164) * math.sqrt(k * (k + 1) / (6.0 * N))
print(f"Nemenyi critical difference (alpha=0.05, k={k}, N={N}): {CD:.3f}")

srt = ranks.sort_values()
fig, ax = plt.subplots(figsize=(COL2, 1.9))
lo, hi = 0.7, k + 0.3
ax.set_xlim(hi, lo); ax.set_ylim(0, 1)
ax.axis("off")
ax.plot([lo, hi], [0.82, 0.82], color=PAL["dark"], linewidth=0.9)
for t in range(1, k + 1):
    ax.plot([t, t], [0.82, 0.86], color=PAL["dark"], linewidth=0.7)
    ax.text(t, 0.89, str(t), ha="center", fontsize=7.5)

half = math.ceil(len(srt) / 2)
for i, (name, r) in enumerate(srt.items()):
    side = i < half
    y = 0.68 - 0.09 * (i if side else i - half)
    xend = lo + 0.25 if side else hi - 0.25
    ax.plot([r, r], [0.82, y], color=METHOD_COLOR.get(name, PAL["gray"]), linewidth=0.9)
    ax.plot([r, xend], [y, y], color=METHOD_COLOR.get(name, PAL["gray"]), linewidth=0.9)
    ax.text(xend + (0.05 if side else -0.05), y, f"{name} ({r:.2f})",
            ha="left" if side else "right", va="center", fontsize=7.5)

# cliques: groups not significantly different
vals = srt.to_numpy(); names = list(srt.index)
yb = 0.79
for i in range(len(vals)):
    j = i
    while j + 1 < len(vals) and vals[j + 1] - vals[i] < CD:
        j += 1
    if j > i:
        ax.plot([vals[i] - 0.02, vals[j] + 0.02], [yb, yb], color=PAL["dark"], linewidth=2.2)
        yb -= 0.035
ax.plot([hi - 0.05, hi - 0.05 - CD], [0.97, 0.97], color=PAL["dark"], linewidth=1.4)
ax.text(hi - 0.05 - CD / 2, 0.985, f"CD = {CD:.2f}", ha="center", fontsize=7.5)
savefig(fig, "fig06_critical_difference", tight=False)


In [ ]:
# ============================================================
# 26 | FIGURE 7 - CALIBRATION BUDGET CURVE
# ============================================================
crows = []
for r in recs:
    c = r.get("config", {})
    if c.get("kind") != "calibration":
        continue
    lbl = PRETTY.get(c["method"], c["method"]) + (" (head-only)" if c.get("head_only") else "")
    for row in r["rows"]:
        crows.append({"method": lbl, "seed": c["seed"], **row})
CAL = pd.DataFrame(crows)

if len(CAL):
    fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.4))

    ax = axes[0]
    for lbl, g in CAL.groupby("method"):
        agg = g.groupby("budget")["bacc"].agg(["mean", "std", "count"]).reset_index()
        se = agg["std"] / np.sqrt(agg["count"])
        col = METHOD_COLOR.get(lbl.split(" ")[0], PAL["gray"])
        ls = "--" if "head-only" in lbl else "-"
        ax.plot(agg["budget"], agg["mean"], color=col, linestyle=ls, marker="o",
                markersize=3, label=lbl)
        ax.fill_between(agg["budget"], agg["mean"] - 1.96 * se, agg["mean"] + 1.96 * se,
                        color=col, alpha=0.15, linewidth=0)
    ax.axhline(1 / CFG.N_CLASSES, color=PAL["gray"], linestyle=":", linewidth=0.9)
    ax.set_xlabel("Labelled target windows"); ax.set_ylabel("Balanced accuracy")
    ax.set_title("(a) Calibration budget", loc="left")
    ax.legend()

    ax = axes[1]
    base = CAL[CAL.budget == 0].groupby(["method", "seed", "subject"])["bacc"].mean()
    gains = []
    for k in sorted(CAL.budget.unique()):
        if k == 0:
            continue
        cur = CAL[CAL.budget == k].groupby(["method", "seed", "subject"])["bacc"].mean()
        d = (cur - base).dropna()
        gains.append({"budget": k, "gain": d.mean(), "se": d.std() / np.sqrt(len(d))})
    G = pd.DataFrame(gains)
    ax.bar(np.arange(len(G)), G["gain"], yerr=1.96 * G["se"], color=PAL["teal"],
           capsize=2, linewidth=0, error_kw=dict(linewidth=0.8, ecolor=PAL["dark"]))
    ax.set_xticks(np.arange(len(G))); ax.set_xticklabels(G["budget"].astype(int))
    ax.axhline(0, color=PAL["dark"], linewidth=0.7)
    ax.set_xlabel("Labelled target windows"); ax.set_ylabel("Gain over zero-shot")
    ax.set_title("(b) Marginal value of calibration", loc="left")
    savefig(fig, "fig07_calibration_budget")

    summ = CAL.groupby(["method", "budget"])["bacc"].agg(["mean", "std"]).round(3).reset_index()
    display(summ)
    to_latex(summ, "tab03_calibration",
             "Balanced accuracy as a function of labelled windows from the target subject.",
             "tab:calib")

    # is the smallest non-zero budget already significantly better than zero-shot?
    for meth in CAL["method"].unique():
        b0 = CAL[(CAL.method == meth) & (CAL.budget == 0)].set_index(["seed", "subject"])["bacc"]
        for k in sorted(CAL.budget.unique())[1:3]:
            bk = CAL[(CAL.method == meth) & (CAL.budget == k)].set_index(["seed", "subject"])["bacc"]
            common = b0.index.intersection(bk.index)
            if len(common) > 5:
                _, p = st.wilcoxon(bk.loc[common], b0.loc[common])
                print(f"{meth:22s} budget {k:3d} vs 0: "
                      f"delta={bk.loc[common].mean()-b0.loc[common].mean():+.4f}, p={p:.3e}")
else:
    print("no calibration records yet")


In [ ]:
# ============================================================
# 27 | FIGURE 8 - ABLATION GRID  (aggregated, split into two figures)
# ============================================================
abl = []
for r in recs:
    c = r.get("config", {})
    t = c.get("tag", "")
    if c.get("proto") != CFG.PROTOCOL or not t.startswith("abl") or "mean" not in r:
        continue
    abl.append({"tag": t, "method": c.get("method"), "seed": c.get("seed"),
                "bacc": r["mean"]["bacc"], "diverged": bool(r.get("diverged", False)),
                "config": c})
ABL = pd.DataFrame(abl)


def _sortkey(v):
    try:
        return (0, float(v), "")
    except (TypeError, ValueError):
        return (1, 0.0, str(v))


def _panel(ax, pref, title, keyfn, colour):
    sub = ABL[ABL.tag.str.startswith(pref)].copy()
    sub["x"] = sub["config"].apply(keyfn)

    # collapse the ABL_REPS replicates into one bar per setting: mean across
    # replicates, error bar = SEM across replicates, not SD across subjects
    g = sub.groupby("x", sort=False).agg(
        m=("bacc", "mean"), s=("bacc", "std"), n=("bacc", "size"),
        div=("diverged", "any")).reset_index()
    g["se"] = g["s"].fillna(0) / np.sqrt(g["n"].clip(lower=1))
    g = g.sort_values("x", key=lambda s: s.map(_sortkey)).reset_index(drop=True)

    xs = np.arange(len(g))
    ax.bar(xs, g["m"], yerr=g["se"], color=colour, linewidth=0, capsize=2,
           width=0.6, error_kw=dict(linewidth=0.7, ecolor=PAL["dark"]))

    # mark settings where at least one replicate blew up
    for i, row in g.iterrows():
        if row["div"]:
            ax.text(i, row["m"] + row["se"] + 0.004, "!", ha="center", va="bottom",
                    fontsize=8, color=PAL["coral"], fontweight="bold")

    labs = [f"{v:g}" if isinstance(v, (int, float, np.floating)) else str(v)
            for v in g["x"]]
    rot = 0 if max((len(l) for l in labs), default=1) <= 5 else 30
    ax.set_xticks(xs)
    ax.set_xticklabels(labs, rotation=rot, fontsize=7,
                       ha="right" if rot else "center")
    ax.set_xlim(-0.6, len(g) - 0.4)
    ax.axhline(1 / CFG.N_CLASSES, color=PAL["gray"], linestyle=":", linewidth=0.8)

    lo = float((g["m"] - g["se"]).min())
    hi = float((g["m"] + g["se"]).max())
    span = max(hi - lo, 0.02)
    ax.set_ylim(max(0.0, lo - 0.35 * span), min(1.02, hi + 0.35 * span))
    ax.set_title(title, loc="left", pad=4, fontsize=9)
    return g


if len(ABL):
    # ---- Figure 8a: optimisation ablations, 2x2 ----
    optim = [("abl_le",   "(a) Local epochs",     lambda c: c["local_epochs"]),
             ("abl_cf",   "(b) Client fraction",  lambda c: c["client_frac"]),
             ("abl_mu",   "(c) FedProx $\\mu$",   lambda c: c["mu"]),
             ("abl_norm", "(d) Per-client norm.", lambda c: "on" if c["normalize"] else "off")]
    optim = [p for p in optim if ABL.tag.str.startswith(p[0]).any()]

    if optim:
        fig, axes = plt.subplots(2, 2, figsize=(COL2, 4.2), constrained_layout=True)
        fig.set_constrained_layout_pads(w_pad=0.10, h_pad=0.14, wspace=0.10, hspace=0.14)
        axes = axes.ravel()
        for i, (ax, spec) in enumerate(zip(axes, optim)):
            _panel(ax, spec[0], spec[1], spec[2], PAL_SEQ[i % len(PAL_SEQ)])
            ax.set_ylabel("Balanced acc." if i % 2 == 0 else "")
        for ax in axes[len(optim):]:
            ax.axis("off")
        savefig(fig, "fig08a_ablations_optimisation", tight=False)

    # ---- Figure 8b: deployment ablations, 1x2 ----
    deploy = [("abl_dp",   "(a) Update noise $\\sigma$", lambda c: c["dp_sigma"]),
              ("abl_arch", "(b) Architecture",           lambda c: c["model"])]
    deploy = [p for p in deploy if ABL.tag.str.startswith(p[0]).any()]

    if deploy:
        fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.6), constrained_layout=True)
        fig.set_constrained_layout_pads(w_pad=0.10, h_pad=0.10, wspace=0.10)
        axes = np.atleast_1d(axes)
        for i, (ax, spec) in enumerate(zip(axes, deploy)):
            _panel(ax, spec[0], spec[1], spec[2], PAL_SEQ[(i + 4) % len(PAL_SEQ)])
            ax.set_ylabel("Balanced acc." if i == 0 else "")
        for ax in axes[len(deploy):]:
            ax.axis("off")
        savefig(fig, "fig08b_ablations_deployment", tight=False)

    # ---- aggregated table ----
    rows = []
    for pref, title, keyfn in optim + deploy:
        sub = ABL[ABL.tag.str.startswith(pref)].copy()
        sub["x"] = sub["config"].apply(keyfn)
        g = sub.groupby("x", sort=False).agg(
            m=("bacc", "mean"), s=("bacc", "std"), n=("bacc", "size"),
            div=("diverged", "any"), meth=("method", "first")).reset_index()
        g = g.sort_values("x", key=lambda s: s.map(_sortkey))
        for r_ in g.itertuples():
            rows.append({"Ablation": title[4:] if title.startswith("(") else title,
                         "Setting": f"{r_.x:g}" if isinstance(r_.x, (int, float, np.floating))
                                    else str(r_.x),
                         "Method": PRETTY.get(r_.meth, r_.meth),
                         "Reps": int(r_.n),
                         "bAcc": round(float(r_.m), 4),
                         "SEM": round(float(r_.s / np.sqrt(max(r_.n, 1))) if r_.n > 1 else 0.0, 4),
                         "Diverged": "yes" if r_.div else ""})
    ABL_TAB = pd.DataFrame(rows)
    display(ABL_TAB)
    to_latex(ABL_TAB, "tab04_ablations",
             f"Ablation study. Balanced accuracy on held-out target subjects, mean over "
             f"{CFG.ABL_REPS} split replicates with standard error across replicates. "
             "Settings marked as diverged produced non-finite parameters in at least "
             "one replicate.", "tab:abl", float_fmt="%.4f")
else:
    print("no ablation records yet")

In [ ]:
# ============================================================
# 28 | HETEROGENEITY VS PERSONALIZATION GAIN  (v2, partial correlation)
# ============================================================

def partial_spearman(x, y, z):
    # Spearman between x and y with the linear effect of z removed from the ranks
    x, y, z = map(lambda a: np.asarray(a, dtype=float), (x, y, z))
    ok = ~(np.isnan(x) | np.isnan(y) | np.isnan(z))
    x, y, z = x[ok], y[ok], z[ok]
    if len(x) < 6:
        return float("nan"), float("nan"), len(x)
    rx, ry, rz = st.rankdata(x), st.rankdata(y), st.rankdata(z)
    ex = rx - np.polyval(np.polyfit(rz, rx, 1), rz)
    ey = ry - np.polyval(np.polyfit(rz, ry, 1), rz)
    r, p = st.pearsonr(ex, ey)
    return float(r), float(p), len(x)


crows = []
for r in recs:
    c = r.get("config", {})
    if c.get("proto") != CFG.PROTOCOL or c.get("kind") != "calibration":
        continue
    lbl = PRETTY.get(c["method"], c["method"]) + (" (head-only)" if c.get("head_only") else "")
    for row in r["rows"]:
        crows.append({"method": lbl, "seed": c["seed"], **row})
CAL = pd.DataFrame(crows)

if len(CAL):
    kmax = max(CFG.CALIB_BUDGETS)
    base = CAL[CAL.budget == 0].groupby("subject")["bacc"].mean()
    topb = CAL[CAL.budget == kmax].groupby("subject")["bacc"].mean()
    GAIN = pd.DataFrame({"subject": base.index, "zeroshot": base.values,
                         "gain": (topb - base).reindex(base.index).values}).dropna()
else:
    GAIN = pd.DataFrame({"subject": piv.index, "zeroshot": piv[REF].values,
                         "gain": (piv[REF] - piv["FedAvg"]).values}) \
        if "FedAvg" in piv.columns else pd.DataFrame()

if len(GAIN):
    M = GAIN.merge(HET, on="subject", how="left")
    predictors = [c for c in ["hbo_amplitude", "hbo_latency_s", "peak_cohen_d",
                              "signal_scale", "mean_shift", "snr_proxy", "within_var"]
                  if c in M.columns and M[c].nunique(dropna=True) > 2]

    srows = []
    for p_ in predictors:
        d = M[[p_, "gain", "zeroshot"]].dropna()
        if len(d) < 8:
            continue
        r0, p0 = st.spearmanr(d[p_], d["gain"])
        # v1 reported r0 only. Subjects with poor zero-shot accuracy gain more by
        # construction, so the raw correlation is confounded with regression to the
        # mean. The partial coefficient removes zero-shot accuracy from both ranks.
        r1, p1, n = partial_spearman(d[p_], d["gain"], d["zeroshot"])
        srows.append({"Descriptor": p_, "rho (raw)": round(r0, 3), "p (raw)": f"{p0:.3f}",
                      "rho (partial)": round(r1, 3), "p (partial)": f"{p1:.3f}",
                      "n": n,
                      "Survives": "yes" if (p1 < 0.05 and abs(r1) > 0.2) else "no"})
    HCORR = pd.DataFrame(srows).sort_values("rho (partial)", key=abs, ascending=False)
    display(HCORR)
    to_latex(HCORR, "tab05_heterogeneity_corr",
             "Association between client hemodynamic descriptors and personalization "
             "gain, before and after partialling out zero-shot accuracy.",
             "tab:het", float_fmt="%.3f")

    # sanity check on the confound itself
    rz, pz = st.spearmanr(M["zeroshot"], M["gain"])
    print(f"\nregression-to-the-mean check: Spearman(zero-shot, gain) "
          f"rho={rz:.3f}, p={pz:.3e}")
    if abs(rz) > 0.4:
        print("  strong. Report partial coefficients only; the raw ones are not interpretable.")

    survivors = HCORR[HCORR["Survives"] == "yes"]["Descriptor"].tolist()
    show = survivors[:3] if survivors else HCORR["Descriptor"].tolist()[:3]
    fig, axes = plt.subplots(1, max(1, len(show)), figsize=(COL2, 2.2),
                             constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, p_ in zip(axes, show):
        d = M[[p_, "gain", "zeroshot"]].dropna()
        sc = ax.scatter(d[p_], d["gain"], s=18, c=d["zeroshot"], cmap="viridis",
                        edgecolor="white", linewidth=0.4)
        if len(d) > 3:
            z = np.polyfit(d[p_], d["gain"], 1)
            xs = np.linspace(d[p_].min(), d[p_].max(), 50)
            ax.plot(xs, np.polyval(z, xs), color=PAL["coral"], linewidth=1.1)
        r1, p1, _ = partial_spearman(d[p_], d["gain"], d["zeroshot"])
        ax.axhline(0, color=PAL["gray"], linewidth=0.7)
        ax.set_title(f"{p_}\npartial " + r"$\rho$" + f"={r1:.2f}, p={p1:.3f}",
                     loc="left", fontsize=8)
        ax.set_xlabel(p_)
    axes[0].set_ylabel("Personalization gain")
    fig.colorbar(sc, ax=axes[-1], fraction=0.046, label="zero-shot bAcc")
    savefig(fig, "fig09_heterogeneity_vs_gain", tight=False)
else:
    print("run the calibration experiment first")


In [ ]:
# ============================================================
# 29 | FAIRNESS AUDIT  (v2, subject-level, cleaned categories)
# ============================================================
DEMO_PATH = ("/kaggle/input/datasets/syedmuntasinfayaz/fnirs2mw-dataset/"
             "fNIRS2MW_dataset/subjects_information.csv")

ID_RE   = re.compile(r"\b(subject|participant|subj|sub)\b|sub_id|_id\b|^id$", re.I)
ATTR_RE = re.compile(r"\b(age|gender|sex_at_birth|race|ethnic|ethnicity|handed|handedness)\b", re.I)

DEMO = None
if os.path.exists(DEMO_PATH):
    DEMO = pd.read_csv(DEMO_PATH)
    DEMO.columns = [str(c).strip() for c in DEMO.columns]
    print("columns:", list(DEMO.columns))

if DEMO is not None:
    idcol = next((c for c in DEMO.columns if ID_RE.search(c)), DEMO.columns[0])
    nums = DEMO[idcol].astype(str).str.extract(r"(\d+)")[0]
    DEMO["subject"] = nums.apply(lambda v: f"sub_{int(v):03d}" if pd.notna(v) else np.nan)
    DEMO = DEMO.dropna(subset=["subject"]).drop_duplicates("subject")

    def clean_cat(v):
        # v1 treated "asian" and "asian,not hispanic or latino or spanish origin"
        # as separate levels, fragmenting an already tiny sample
        if pd.isna(v):
            return np.nan
        s = str(v).strip().lower()
        if s in ("", "nan", "na", "n/a", "prefer not to say", "none"):
            return np.nan
        parts = [p.strip() for p in re.split(r"[,;/]", s) if p.strip()]
        head = parts[0]
        if "hispanic" in head or "latino" in head or "spanish" in head:
            return "hispanic/latino"
        for key in ["asian", "white", "black", "african", "native", "pacific",
                    "middle eastern", "multi", "other"]:
            if key in head:
                return "black/african" if key in ("black", "african") else key
        return head[:24]

    attrs = [c for c in DEMO.columns if ATTR_RE.search(c) and c != "subject"]
    print("candidate attributes:", attrs)

    # subject-level means: one row per subject, averaged over the replicates in
    # which it was a target. v1 audited 42 subject-by-seed rows.
    perf = piv[REF].rename("bacc").reset_index()
    FA = perf.merge(DEMO[["subject"] + attrs], on="subject", how="left")
    print(f"audited: {len(FA)} subjects, "
          f"{FA[attrs].notna().any(axis=1).sum()} with demographics")

    frows = []
    for a in attrs:
        col = FA[a]
        if pd.api.types.is_numeric_dtype(col) and col.nunique(dropna=True) > 5:
            grp = pd.qcut(col, 2, labels=["lower", "upper"], duplicates="drop")
            gname = f"{a} (median split)"
        else:
            grp = col.map(clean_cat)
            gname = a
        valid = grp.notna()
        levels_named = [(str(l), g) for l, g in FA[valid].groupby(grp[valid], observed=True)
                        if len(g) >= 4]
        if len(levels_named) < 2:
            print(f"[drop] {gname}: only {len(levels_named)} usable level(s) with n>=4")
            continue
        for lvl, g in levels_named:
            lo, hi = bootstrap_ci(g["bacc"].to_numpy())
            frows.append({"Attribute": gname, "Group": lvl, "n": int(len(g)),
                          "bAcc": round(float(g["bacc"].mean()), 3),
                          "95% CI": f"[{lo:.3f}, {hi:.3f}]"})
        arrays = [g["bacc"].to_numpy() for _, g in levels_named]
        h, p = st.kruskal(*arrays)
        gap = max(a_.mean() for a_ in arrays) - min(a_.mean() for a_ in arrays)
        # smallest group size drives the power; report it so the result is readable
        print(f"{gname}: H={h:.2f}, p={p:.3f}, k={len(arrays)}, "
              f"min n={min(len(a_) for a_ in arrays)}, gap={gap:.3f}"
              + ("   [UNDERPOWERED, descriptive only]"
                 if min(len(a_) for a_ in arrays) < 10 else ""))

    FAIR = pd.DataFrame(frows)
    if len(FAIR):
        display(FAIR)
        to_latex(FAIR, "tab06_fairness",
                 f"Subgroup performance of {REF} on {piv.shape[0]} held-out target "
                 "subjects with bootstrap 95\\% confidence intervals. Groups with "
                 "fewer than four subjects are omitted. Intervals are wide and the "
                 "audit is descriptive, not a test of the absence of bias.",
                 "tab:fair")

        fig, ax = plt.subplots(figsize=(COL2, max(2.0, 0.32 * len(FAIR) + 0.8)),
                               constrained_layout=True)
        ypos = np.arange(len(FAIR))
        uattr = list(dict.fromkeys(FAIR["Attribute"]))
        cmap = {a: PAL_SEQ[i % len(PAL_SEQ)] for i, a in enumerate(uattr)}
        ax.barh(ypos, FAIR["bAcc"], color=[cmap[a] for a in FAIR["Attribute"]],
                linewidth=0, height=0.62)
        cis = FAIR["95% CI"].str.strip("[]").str.split(",", expand=True).astype(float)
        ax.hlines(ypos, cis[0], cis[1], color=PAL["dark"], linewidth=0.9)
        ax.axvline(FA["bacc"].mean(), color=PAL["coral"], linestyle="--",
                   linewidth=1.0, label="overall mean")
        ax.axvline(1 / CFG.N_CLASSES, color=PAL["gray"], linestyle=":",
                   linewidth=0.9, label="chance")
        ax.set_yticks(ypos)
        ax.set_yticklabels([f"{r.Attribute}: {r.Group} (n={r.n})" for r in FAIR.itertuples()],
                           fontsize=7)
        ax.invert_yaxis(); ax.set_xlabel("Balanced accuracy")
        ax.set_xlim(max(0.0, min(cis[0].min(), 1 / CFG.N_CLASSES) - 0.05),
                    min(1.02, cis[1].max() + 0.03))
        ax.legend(loc="lower right")
        savefig(fig, "fig10_fairness_audit", tight=False)
    else:
        print("No subgroup reached n>=4. Report the audit as underpowered.")
else:
    print("No demographics file at", DEMO_PATH)


In [ ]:
# ============================================================
# 30 | FIGURE 11 - CONFUSION MATRIX AND ROC FOR THE BEST METHOD
# ============================================================
bundle = load_best_bundle(REF.lower(), CFG.SEEDS[0])
if bundle is not None:
    seed_everything(CFG.SEEDS[0])
    _, _, te_s = subject_split(CFG.SEEDS[0])
    test_sub = prepare_client_cache(te_s, True)
    model = build_model()
    model.load_state_dict(bundle["global_state"], strict=True)
    if bundle["personal"]:
        model.load_state_dict(bundle["personal"], strict=False)

    Pall, Yall = [], []
    for s, d in test_sub.items():
        _, P, Y = evaluate(model, make_loader(*d["test"], 256, False))
        Pall.append(P); Yall.append(Y)
    P = np.concatenate(Pall); Y = np.concatenate(Yall)

    fig, axes = plt.subplots(1, 2, figsize=(COL2, 2.5))
    cm = confusion_matrix(Y, P.argmax(1), normalize="true")
    im = axes[0].imshow(cm, cmap="Blues", vmin=0, vmax=1)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            axes[0].text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                         color="white" if cm[i, j] > 0.55 else PAL["dark"], fontsize=8)
    axes[0].set_xticks(range(CFG.N_CLASSES)); axes[0].set_yticks(range(CFG.N_CLASSES))
    axes[0].set_xticklabels([f"{l}-back" for l in CFG.TASK_LABELS])
    axes[0].set_yticklabels([f"{l}-back" for l in CFG.TASK_LABELS])
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
    axes[0].set_title(f"(a) {REF} confusion", loc="left")
    axes[0].grid(False)
    fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

    ax = axes[1]
    if CFG.N_CLASSES == 2:
        for s, d in list(test_sub.items()):
            _, Ps, Ys = evaluate(model, make_loader(*d["test"], 256, False))
            if len(np.unique(Ys)) == 2:
                fpr, tpr, _ = roc_curve(Ys, Ps[:, 1])
                ax.plot(fpr, tpr, color=PAL["gray"], alpha=0.35, linewidth=0.7)
        fpr, tpr, _ = roc_curve(Y, P[:, 1])
        ax.plot(fpr, tpr, color=PAL["blue"], linewidth=1.6,
                label=f"pooled AUROC = {roc_auc_score(Y, P[:,1]):.3f}")
        ax.plot([0, 1], [0, 1], color=PAL["dark"], linestyle=":", linewidth=0.8)
        ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
        ax.set_title("(b) ROC, thin lines = individual subjects", loc="left")
        ax.legend(loc="lower right")
    savefig(fig, "fig11_confusion_roc")
    del model
else:
    print("no best bundle found; run the main comparison first")


In [ ]:
# ============================================================
# 31 | COMMUNICATION COST ACCOUNTING
# ============================================================
m = build_model()
sizes = {}
for meth, fn in PERSONAL_KEY_FN.items():
    shared = sum(v.numel() for k, v in m.state_dict().items()
                 if not fn(k) and torch.is_floating_point(v))
    total = sum(v.numel() for v in m.state_dict().values() if torch.is_floating_point(v))
    sizes[PRETTY[meth]] = {"shared_params": shared, "total_params": total,
                           "MB_per_client_round": shared * 4 / 1e6,
                           "shared_frac": shared / total}
del m
COMM = pd.DataFrame(sizes).T.reset_index().rename(columns={"index": "Method"})
COMM["MB_total_uplink"] = (COMM["MB_per_client_round"] *
                           max(1, int(CFG.CLIENT_FRAC * len(TRAIN_S))) * CFG.ROUNDS)
COMM = COMM.round(4)
display(COMM)
to_latex(COMM, "tab07_communication",
         "Per-round uplink cost. FedPer and FedBN transmit only the shared block.",
         "tab:comm", float_fmt="%.3f")


In [ ]:
# ============================================================
# 32 | PACKAGE EVERYTHING FOR THE MANUSCRIPT
# ============================================================
manifest = {
    "generated":      time.strftime("%Y-%m-%d %H:%M:%S"),
    "n_subjects":     len(SUBJ_IDS),
    "n_windows":      int(sum(v["X"].shape[0] for v in SUBJECTS.values())),
    "n_channels":     int(N_CH),
    "task_labels":    CFG.TASK_LABELS,
    "seeds":          CFG.SEEDS,
    "train_clients":  len(TRAIN_S),
    "val_subjects":   len(VAL_S),
    "test_subjects":  len(TEST_S),
    "guard_chunks":   CFG.GUARD_CHUNKS,
    "rounds":         CFG.ROUNDS,
    "figures":        sorted(p.name for p in Path(CFG.FIG_DIR).glob("*.pdf")),
    "tables":         sorted(p.name for p in Path(CFG.TAB_DIR).glob("*")),
    "n_experiments":  len(list(Path(CFG.RESULT_DIR).glob("*.json"))),
    "torch":          torch.__version__,
    "device":         str(DEVICE),
}
with open(f"{CFG.WORK_DIR}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

bundle_dir = Path(CFG.WORK_DIR) / "paper_bundle"
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir()
for src in [CFG.FIG_DIR, CFG.TAB_DIR, CFG.RESULT_DIR]:
    shutil.copytree(src, bundle_dir / Path(src).name)
shutil.copy(f"{CFG.WORK_DIR}/manifest.json", bundle_dir / "manifest.json")
shutil.make_archive(str(Path(CFG.WORK_DIR) / "paper_bundle"), "zip", bundle_dir)

print(json.dumps(manifest, indent=2))
print("\npackaged -> /kaggle/working/paper_bundle.zip")


In [ ]:
# ============================================================
# REVISION ANALYSES for fed-workload-nirs_2.ipynb
# Cell-based. Paste each "# %%" block as a new cell AFTER cell 31
# of the notebook, in the same Kaggle session (it reuses PS, piv,
# CAL, HET, SUBJECTS, run_federated, load_best_bundle, etc.).
#
# Sections A, B, C, D, H: analysis only, no training.
# Sections E, F, G: train or fit; all resumable via cfg_hash /
# has_result / save_result, so a restart skips finished work.
# Nothing here has been run. Check the printed numbers against
# the manuscript before you use them.
# ============================================================

# %% 0 | setup
REV_DIR = Path(CFG.WORK_DIR) / "revision"
REV_DIR.mkdir(exist_ok=True)
STRIDE = 3                       # samples between consecutive window ids
MARGINS = [0.03, 0.05, 0.10]     # equivalence margins in balanced accuracy


# %% A | test-block composition, spans, independent segments (NO training)
# Answers: class counts per partition (reviewer Major 2), single-class test
# blocks, NaN AUROC counts, and what a window budget means in seconds.
def raw_chunks(sid):
    z = np.load(Path(CFG.CACHE_DIR) / f"{sid}.npz", allow_pickle=True)
    m = np.isin(z["y"], CFG.TASK_LABELS)
    return z["chunk"][m]

rows = []
for sid in SUBJ_IDS:
    tr, va, te = temporal_split(sid)
    y = SUBJECTS[sid]["y"]
    try:
        ch = raw_chunks(sid).astype(float)
        steps = np.diff(ch)
        max_gap, n_gaps = float(steps.max()), int((steps != 1).sum())
    except Exception:
        max_gap, n_gaps = np.nan, -1
    rows.append(dict(
        subject=sid, n=len(y),
        tr_n=len(tr), tr_pos=int(y[tr].sum()),
        va_n=len(va),
        te_n=len(te), te_neg=int((y[te] == 0).sum()), te_pos=int((y[te] == 1).sum()),
        te_runs=int((np.diff(y[te]) != 0).sum()) + 1,
        te_span_s=(STRIDE * (len(te) - 1) + CFG.N_TIMESTEPS) / FS,
        max_chunk_gap=max_gap, n_chunk_gaps=n_gaps))
COMP = pd.DataFrame(rows)
COMP["te_single_class"] = (COMP.te_neg == 0) | (COMP.te_pos == 0)
COMP.to_csv(REV_DIR / "partition_composition.csv", index=False)

print("subjects with a single-class test block:", int(COMP.te_single_class.sum()), "/", len(COMP))
print(COMP[["tr_n", "tr_pos", "te_n", "te_neg", "te_pos", "te_runs", "te_span_s"]].describe().round(2))
print("chunk ids not consecutive (n_chunk_gaps > 0 means stride-based spans below are approximate):",
      int((COMP.n_chunk_gaps != 0).sum()), "subjects")
print("\nundefined AUROC per method (rows with NaN):")
print(PS.groupby("method")["auroc"].apply(lambda s: int(s.isna().sum())).reindex(ORDER))


# %% B | subject-level tables, paired AUROC, TOST, all pairs, paired dominance (NO training)
def paired_dominance(d):
    d = np.asarray(d, float)
    return float((np.sum(d > 0) - np.sum(d < 0)) / len(d))

def tost_wilcoxon(d, margin):
    d = np.asarray(d, float)
    p_lo = st.wilcoxon(d + margin, alternative="greater", zero_method="wilcox").pvalue
    p_hi = st.wilcoxon(d - margin, alternative="less", zero_method="wilcox").pvalue
    return float(max(p_lo, p_hi))

SUBJ = (PS.groupby(["method", "subject"])[["bacc", "auroc", "f1", "kappa"]]
          .mean().reset_index())
tab_subj = (SUBJ.groupby("method")[["bacc", "auroc", "f1", "kappa"]]
                .agg(["mean", "std"]).reindex(ORDER).round(3))
print("Subject-level means (N subjects per method for AUROC shown below):")
display(tab_subj)
print(SUBJ.groupby("method")["auroc"].count().reindex(ORDER))
tab_subj.to_csv(REV_DIR / "table2_subject_level.csv")

def compare(metric, ref, others, pivot):
    out = []
    for m in others:
        d = (pivot[ref] - pivot[m]).dropna()
        lo95, hi95 = bootstrap_ci(d.to_numpy(), alpha=0.05)
        lo90, hi90 = bootstrap_ci(d.to_numpy(), alpha=0.10)
        try:
            p = st.wilcoxon(d, zero_method="wilcox").pvalue
        except ValueError:
            p = 1.0
        row = dict(metric=metric, comparison=f"{ref} vs {m}", n=len(d), delta=d.mean(),
                   ci95_lo=lo95, ci95_hi=hi95, ci90_lo=lo90, ci90_hi=hi90, p=p,
                   dominance=paired_dominance(d))
        for mg in MARGINS:
            try:
                row[f"tost_p_{mg}"] = tost_wilcoxon(d, mg)
            except ValueError:
                row[f"tost_p_{mg}"] = np.nan
        out.append(row)
    return pd.DataFrame(out)

others = [m for m in methods_present if m != REF]
EQ_BACC = compare("bacc", REF, others, piv)
piv_auc = SUBJ.pivot(index="subject", columns="method", values="auroc")
EQ_AUC = compare("auroc", REF, others, piv_auc)
EQ = pd.concat([EQ_BACC, EQ_AUC], ignore_index=True)
EQ["holm_p_8"] = np.nan
for metric in ("bacc", "auroc"):
    sel = EQ.metric == metric
    adj_ = holm(EQ.loc[sel, "p"].to_numpy(), EQ.loc[sel, "comparison"].tolist())
    EQ.loc[sel, "holm_p_8"] = EQ.loc[sel, "comparison"].map(adj_)
display(EQ.round(4))
EQ.to_csv(REV_DIR / "equivalence_vs_ref.csv", index=False)
print("\nEquivalence at a margin holds when the TOST p < 0.05 (or the 90% CI lies inside +/- margin).")

# all 36 pairs, Holm over the full family
names, ps_ = [], []
for a_, b_ in itertools.combinations(methods_present, 2):
    d = (piv[a_] - piv[b_]).to_numpy()
    try:
        p = st.wilcoxon(d, zero_method="wilcox").pvalue
    except ValueError:
        p = 1.0
    names.append(f"{a_} vs {b_}"); ps_.append(p)
adj36 = holm(np.array(ps_), names)
ALL36 = pd.DataFrame({"pair": names, "p": ps_, "holm_p_36": [adj36[n] for n in names]})
print(f"\nall {len(names)} pairs: smallest Holm p = {ALL36.holm_p_36.min():.3f}")
ALL36.to_csv(REV_DIR / "all_pairs_holm.csv", index=False)


# %% C | calibration: per-budget tests, subject count, true spans (NO training)
cal_ref = CAL[CAL.method == "FedBN"]
print("calibration replicates:", cal_ref.seed.nunique(),
      "| distinct target subjects:", cal_ref.subject.nunique(),
      "| subject x replicate observations:", len(cal_ref[cal_ref.budget == 0]))

sub_cal = cal_ref.groupby(["budget", "subject"])["bacc"].mean().unstack(0)
zs = sub_cal[0]
rows, ps_ = [], []
for k in [b for b in CFG.CALIB_BUDGETS if b > 0]:
    d = (sub_cal[k] - zs).dropna()
    lo, hi = bootstrap_ci(d.to_numpy())
    p = st.wilcoxon(d, zero_method="wilcox").pvalue
    rows.append(dict(budget=k, n=len(d), delta=d.mean(), ci_lo=lo, ci_hi=hi, p=p))
    ps_.append(p)
adj = holm(np.array(ps_), [r["budget"] for r in rows])
CALTEST = pd.DataFrame(rows)
CALTEST["holm_p_5"] = CALTEST.budget.map(adj)
display(CALTEST.round(4))
CALTEST.to_csv(REV_DIR / "calibration_tests_fedbn.csv", index=False)

# true span of the class-balanced tail block, mirroring calibration_subset
def cal_indices(ytr, k, n_cls=2):
    per = max(1, k // n_cls)
    taken = {c: [] for c in range(n_cls)}
    for i in range(len(ytr) - 1, -1, -1):
        c = int(ytr[i])
        if len(taken[c]) < per:
            taken[c].append(i)
        if all(len(v) >= per for v in taken.values()):
            break
    return np.sort(np.concatenate([np.asarray(v, int) for v in taken.values() if v]))

def span_s(starts):
    return (starts.max() - starts.min() + CFG.N_TIMESTEPS) / FS

def covered_s(starts):
    s = np.sort(starts); cov = CFG.N_TIMESTEPS
    for a, b in zip(s[:-1], s[1:]):
        cov += min(CFG.N_TIMESTEPS, b - a)
    return cov / FS

win_s = CFG.N_TIMESTEPS / FS
rows = []
for k in [b for b in CFG.CALIB_BUDGETS if b > 0]:
    sp, cv = [], []
    for sid in SUBJ_IDS:
        tr, _, _ = temporal_split(sid)
        y = SUBJECTS[sid]["y"]
        idx = cal_indices(y[tr], k)
        try:
            starts = STRIDE * raw_chunks(sid).astype(float)[tr][idx]
        except Exception:
            starts = STRIDE * idx.astype(float)
        sp.append(span_s(starts)); cv.append(covered_s(starts))
    rows.append(dict(budget=k, mean_span_s=np.mean(sp), mean_covered_s=np.mean(cv),
                     nonoverlap_equiv=np.mean(cv) / win_s))
SPANS = pd.DataFrame(rows).round(2)
display(SPANS)
SPANS.to_csv(REV_DIR / "calibration_spans.csv", index=False)
print("covered_s counts each second of recording once; nonoverlap_equiv = covered_s / window length.")


# %% D | how much bimodality does sampling alone produce? (NO training)
# Crude null: every subject has the same true per-segment accuracy p_true, and
# each contiguous run of one class in the test block is classified right or
# wrong as a unit. Compare the simulated counts with the observed 42 and 22.
def runs_of(y):
    r, start = [], 0
    for i in range(1, len(y) + 1):
        if i == len(y) or y[i] != y[start]:
            r.append((int(y[start]), i - start)); start = i
    return r

target_subs = list(piv.index)
RUNS = {sid: runs_of(SUBJECTS[sid]["y"][temporal_split(sid)[2]]) for sid in target_subs}

def sim_counts(p_true, n_sim=2000, seed=0):
    rng = np.random.RandomState(seed)
    hi_, lo_ = [], []
    for _ in range(n_sim):
        b = []
        for sid in target_subs:
            r = RUNS[sid]
            ok = rng.rand(len(r)) < p_true
            rec = {}
            for (c, L), o in zip(r, ok):
                num, den = rec.get(c, (0, 0))
                rec[c] = (num + L * o, den + L)
            b.append(np.mean([n_ / d_ for n_, d_ in rec.values()]))
        b = np.array(b)
        hi_.append((b > 0.60).sum()); lo_.append((b < 0.55).sum())
    return np.array(hi_), np.array(lo_)

obs_hi = int((piv[REF] > 0.60).sum()); obs_lo = int((piv[REF] < 0.55).sum())
for p_true in (0.60, 0.66, 0.72):
    hi_, lo_ = sim_counts(p_true)
    print(f"p_true={p_true:.2f}: >0.60 mean {hi_.mean():.1f} (95% {np.percentile(hi_,2.5):.0f}-{np.percentile(hi_,97.5):.0f}), "
          f"<0.55 mean {lo_.mean():.1f} (95% {np.percentile(lo_,2.5):.0f}-{np.percentile(lo_,97.5):.0f})")
print(f"observed ({REF}): >0.60 = {obs_hi}, <0.55 = {obs_lo}")
print("If the simulated ranges cover the observed counts, the bimodality claim cannot be separated from short test blocks.")


# %% E | calibration sweep for FedBN (TRAINS; fine-tuning only, resumable)
# Discriminating experiment for the "harmful calibration" claim: does a smaller
# learning rate or fewer epochs remove the degradation? Two bn_eval arms check
# the BN-statistics explanation directly.
SWEEP_BUDGETS = [8, 16, 32, 64, 128]
SWEEP_CFGS = [dict(lr=lr, epochs=ep, bn_eval=False)
              for lr in (1e-5, 1e-4, 5e-4) for ep in (1, 5, 20)]
SWEEP_CFGS += [dict(lr=5e-4, epochs=20, bn_eval=True),
               dict(lr=1e-4, epochs=20, bn_eval=True)]

def finetune(model, X, y, epochs, lr, bn_eval=False, bs=32):
    model.train()
    if bn_eval:
        for m in model.modules():
            if isinstance(m, nn.BatchNorm1d):
                m.eval()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG.WEIGHT_DECAY)
    scaler = make_scaler()
    crit = nn.CrossEntropyLoss()
    loader = make_loader(X, y, min(bs, len(y)), True)
    for _ in range(epochs):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True); yb = yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE.type, enabled=USE_AMP):
                loss = crit(model(xb), yb)
            if not torch.isfinite(loss):
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
            scaler.step(opt); scaler.update()

def run_sweep(method, seed, cfgd, force=False):
    conf = dict(proto="rev-sweep", kind="calibration-sweep", method=method, seed=seed,
                budgets=SWEEP_BUDGETS, **cfgd)
    exp = cfg_hash(conf)
    if has_result(exp) and not force:
        return load_result(exp)
    bundle = load_best_bundle(method, seed)
    if bundle is None:
        print(f"[warn] no bundle for {method} seed={seed}")
        return None
    seed_everything(seed)
    _, _, te_s = subject_split(seed)
    test_sub = prepare_client_cache(te_s, True)
    rows = []
    for k in SWEEP_BUDGETS:
        for s, d in test_sub.items():
            model = build_model()
            model.load_state_dict(bundle["global_state"], strict=True)
            if bundle["personal"]:
                model.load_state_dict(bundle["personal"], strict=False)
            Xk, yk = calibration_subset(*d["train"], k, CFG.N_CLASSES)
            finetune(model, Xk, yk, cfgd["epochs"], cfgd["lr"], cfgd["bn_eval"])
            m, _, _ = evaluate(model, make_loader(*d["test"], 256, False))
            rows.append(dict(budget=k, subject=s, **m))
            del model
    rec = dict(exp=exp, config=conf, rows=rows)
    save_result(exp, rec)
    print(f"[done] sweep {method} seed={seed} {cfgd}")
    return rec

for sd in CFG.SEEDS[:6]:                      # same six replicates as the calibration experiment
    for cfgd in SWEEP_CFGS:
        run_sweep("fedbn", sd, cfgd)

def summarize_sweep():
    recs = [r for r in all_results() if r.get("config", {}).get("proto") == "rev-sweep"]
    rows = []
    for r in recs:
        c = r["config"]
        for row in r["rows"]:
            rows.append(dict(lr=c["lr"], epochs=c["epochs"], bn_eval=c["bn_eval"],
                             seed=c["seed"], **row))
    SW = pd.DataFrame(rows)
    zs_ = CAL[(CAL.method == "FedBN") & (CAL.budget == 0)].groupby("subject")["bacc"].mean()
    g = SW.groupby(["lr", "epochs", "bn_eval", "budget", "subject"])["bacc"].mean().reset_index()
    out = []
    for key, gg in g.groupby(["lr", "epochs", "bn_eval", "budget"]):
        s = gg.set_index("subject")["bacc"]
        d = (s - zs_.reindex(s.index)).dropna()
        out.append(dict(lr=key[0], epochs=key[1], bn_eval=key[2], budget=key[3],
                        delta_vs_zeroshot=d.mean(), p=st.wilcoxon(d).pvalue, n=len(d)))
    return pd.DataFrame(out)

SWEEP = summarize_sweep()
SWEEP.to_csv(REV_DIR / "calibration_sweep.csv", index=False)
display(SWEEP.pivot_table(index=["lr", "epochs", "bn_eval"], columns="budget",
                          values="delta_vs_zeroshot").round(3))
print("Read: if small lr or few epochs give delta near 0 or positive, 'harmful' is configuration-specific.")
print("The row lr=5e-4, epochs=20, bn_eval=False should reproduce Table 4 (FedBN) up to rounding.")


# %% F | normalisation OFF for FedAvg and FedAff (TRAINS; resumable)
# Tests whether the client affine block can help once per-channel scale
# heterogeneity is left in the input. Tag starts with "abl" so the main tables
# (cell 21) ignore these records.
for meth in ["fedavg", "fedaff"]:
    for sd in CFG.SEEDS:
        run_federated(meth, sd, normalize=False, tag="abl_rev_normoff")

rows = []
for r in all_results():
    c = r.get("config", {})
    if c.get("tag") == "abl_rev_normoff" and "per_subject" in r:
        for s, m in r["per_subject"].items():
            rows.append(dict(method=PRETTY[c["method"]] + " (norm off)", seed=c["seed"],
                             subject=s, **m))
NO = pd.DataFrame(rows)
no_sub = NO.pivot_table(index="subject", columns="method", values="bacc")
cmp_ = no_sub.join(piv[["FedAvg", "FedAff"]].rename(columns=lambda c: c + " (norm on)"), how="inner")
print("n subjects:", len(cmp_))
print(cmp_.mean().round(3))
for a_, b_ in [("FedAff (norm off)", "FedAvg (norm off)"),
               ("FedAff (norm on)", "FedAvg (norm on)"),
               ("FedAvg (norm on)", "FedAvg (norm off)"),
               ("FedAff (norm on)", "FedAff (norm off)")]:
    d = (cmp_[a_] - cmp_[b_]).dropna()
    lo, hi = bootstrap_ci(d.to_numpy())
    print(f"{a_} - {b_}: {d.mean():+.4f} [{lo:+.3f}, {hi:+.3f}] p={st.wilcoxon(d).pvalue:.3f}")
cmp_.to_csv(REV_DIR / "norm_off_comparison.csv")


# %% G | corrected classical features (fits only; resumable)
# Replaces "normalized area" (identical to the mean) with the mean absolute first
# difference, and the endpoint-difference slope with a least-squares slope.
# Records use proto "rev-feat2", so cell 21 ignores them.
def window_features_v2(X):
    T = X.shape[2]
    t = np.arange(T, dtype=np.float32) - (T - 1) / 2
    slope = (X * t).sum(2) / (t ** 2).sum()
    mean = X.mean(2); std = X.std(2); p2p = X.max(2) - X.min(2)
    sk = np.nan_to_num(st.skew(X, axis=2)); ku = np.nan_to_num(st.kurtosis(X, axis=2))
    half = T // 2
    dmean = X[:, :, half:].mean(2) - X[:, :, :half].mean(2)
    mad = np.abs(np.diff(X, axis=2)).mean(2)
    return np.concatenate([mean, std, slope, p2p, sk, ku, dmean, mad], axis=1)

def run_classical_v2(seed, kind="lr", normalize=True, force=False):
    conf = dict(proto="rev-feat2", kind="classical", method=f"{kind}-feat2", seed=seed,
                normalize=normalize, labels=CFG.TASK_LABELS)
    exp = cfg_hash(conf)
    if has_result(exp) and not force:
        return load_result(exp)
    seed_everything(seed)
    tr_s, _, te_s = subject_split(seed)
    clients = prepare_client_cache(tr_s, normalize)
    test_sub = prepare_client_cache(te_s, normalize)
    Xtr = window_features_v2(np.concatenate([d["train"][0] for d in clients.values()]))
    ytr = np.concatenate([d["train"][1] for d in clients.values()])
    sc = StandardScaler().fit(Xtr)
    clf = (LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0) if kind == "lr"
           else RandomForestClassifier(n_estimators=400, class_weight="balanced",
                                       n_jobs=-1, random_state=seed, min_samples_leaf=3))
    clf.fit(sc.transform(Xtr), ytr)
    per_subject = {}
    for s, d in test_sub.items():
        prob = clf.predict_proba(sc.transform(window_features_v2(d["test"][0])))
        per_subject[s] = compute_metrics(d["test"][1], prob.argmax(1), prob)
    rec = dict(exp=exp, config=conf, per_subject=per_subject)
    save_result(exp, rec)
    print(f"[done] {kind}-feat2 seed={seed}")
    return rec

for sd in CFG.SEEDS:
    for kind in ("lr", "rf"):
        run_classical_v2(sd, kind)

rows = []
for r in all_results():
    c = r.get("config", {})
    if c.get("proto") == "rev-feat2":
        for s, m in r["per_subject"].items():
            rows.append(dict(method=c["method"], seed=c["seed"], subject=s, **m))
F2 = pd.DataFrame(rows)
f2_sub = F2.pivot_table(index="subject", columns="method", values="bacc")
print(f2_sub.mean().round(3))
print("old LR-feat / RF-feat subject-level:", piv[["LR-feat", "RF-feat"]].mean().round(3).to_dict())
print("Replace LR-feat and RF-feat in Table 2, the Friedman test and Figure 9 with the feat2 rows if you adopt them.")


# %% H | gain definition robustness for Table 5 (NO training)
# The notebook's gain averages (128 windows minus zero-shot) over four calibration
# variants. Check that the partial correlations survive with FedBN alone.
def fedbn_gain(k):
    base = CAL[(CAL.method == "FedBN") & (CAL.budget == 0)].groupby("subject")["bacc"].mean()
    top = CAL[(CAL.method == "FedBN") & (CAL.budget == k)].groupby("subject")["bacc"].mean()
    return pd.DataFrame({"zeroshot": base, "gain": top - base}).dropna().reset_index()

for k in (64, 128):
    G = fedbn_gain(k).merge(HET, on="subject", how="left")
    print(f"\nFedBN-only gain at {k} windows, n={len(G)}")
    for p_ in ["within_var", "signal_scale", "mean_shift", "hbo_amplitude", "snr_proxy"]:
        r1, p1, n = partial_spearman(G[p_], G["gain"], G["zeroshot"])
        print(f"  {p_:14s} partial rho={r1:+.3f} p={p1:.3f} n={n}")
print("\nFor Figure 14, plot ranks or drop the fitted line: one extreme subject dominates the raw-value fit.")